# **Etapa 5: Visualización y Presentación (Dataset FIFA 2015-2024)**

**Curso:** Introducción al Análisis de Datos (IC-8076)

**Proyecto:** Proyecto Programado — De datos a Información | **Etapa:** 5 de 5

**Presentación interactiva:** https://josee-arrietaa-31.github.io/etapa5-fifa/

**Integrantes del Grupo I:**

1. Melany Arrieta
2. Andrés Arrieta
3. Laura Alpízar

## **Objetivo y decisiones de diseño**

> *En esta etapa se crearán dashboards, gráficos y presentaciones para comunicar con eficiencia la información encontrada en las etapas anteriores.*

El enunciado pide tres cosas, y cada una tiene su paso en este notebook:

| Tarea del enunciado | Dónde se resuelve |
|---|---|
| Identificar las preguntas que se respondieron en cada una de las secciones. | **Paso 2:** inventario de las preguntas de las Etapas 1 a 4, con su respuesta. |
| Para cada respuesta, elegir cómo representarla: tabla o plot. | **Paso 2** (la elección y su justificación) y **Paso 3** (la representación construida). |
| Dashboards para los 3 primeros experimentos prescriptivos: identificar los valores que el usuario podría ingresar y crear un dashboard interactivo. | **Paso 4:** parámetros de entrada y tres dashboards (transferencias, construcción de equipo, vender o no). |

Las decisiones que afectan a toda la etapa son:

| Decisión | Por qué se tomó |
|---|---|
| **Plotly para los gráficos e ipywidgets para los controles.** | Ambas librerías vienen instaladas en Google Colab, así que el notebook corre sin instalar nada. Plotly permite pasar el cursor para ver el valor exacto, hacer zoom y ocultar series desde la leyenda, algo que un gráfico estático no permite. |
| **Regla fija para elegir entre tabla y gráfico** (ver abajo). | Para que la elección no dependa del gusto de cada integrante y sea la misma en todo el notebook. |
| **Paleta fija por posición y por grupo de edad.** | El mismo color significa lo mismo en todos los gráficos: por ejemplo, los delanteros siempre son rojos y los jóvenes siempre son celestes. |
| **Los dashboards reutilizan la lógica de la Etapa 4.** | Con los valores del enunciado, cada dashboard reproduce el resultado de la Etapa 4 (una celda de verificación imprime ese resultado antes de los controles), y el código comprueba con `assert` que cada recomendación cumpla las restricciones del problema (presupuesto, edad, potencial y jugadores por posición). |
| **El cálculo está separado de la interfaz.** | Cada dashboard llama a una función de cálculo (`resolver_transferencias`, `construir_equipo`, `evaluar_venta`) que se puede probar sin los controles. |
| **Los modelos de la Etapa 4 se reentrenan con las mismas semillas y particiones.** | Para graficarlos (real contra predicho, matrices de confusión, curvas ROC, clusters) hay que tenerlos entrenados; una tabla compara las métricas obtenidas con las reportadas en la Etapa 4. |

**Regla para elegir la representación:**

| Se usa una **tabla** cuando... | Se usa un **gráfico** cuando... |
|---|---|
| la respuesta es una lista de jugadores o clubes que el lector necesita identificar por nombre; | la respuesta compara pocas categorías (barras); |
| importan los valores exactos (costos, métricas de un modelo, tiempos); | la respuesta describe una distribución (histograma, boxplot); |
| se comparan varias medidas a la vez para pocas filas. | la respuesta es una tendencia en el tiempo (líneas), una relación entre variables (dispersión, mapa de calor) o una jerarquía (treemap, sunburst). |

## **Paso 1: Configuración y Carga de Datos**

Se usa el mismo CSV normalizado de las Etapas 2, 3 y 4, con la misma carga y las mismas homologaciones, para que todas las cifras de esta etapa coincidan con las anteriores.

#### **Contenido de la sección:**
1. Librerías, paleta y funciones de apoyo.
2. Conexión a Google Drive.
3. Carga y homologación del dataset normalizado.
4. Preparación del mercado actual (temporada 2024) para los dashboards.

### **1. Librerías, paleta y funciones de apoyo.**

Además de las librerías, se definen la paleta de colores y tres funciones que se usan en todo el notebook: `fmt_eur` (formato de euros), `mostrar_tabla` (tablas con el mismo estilo) y `tarjetas` (indicadores resumidos en los dashboards).

In [1]:
# Manipulación de datos
import os, sys, glob, warnings
import numpy as np
import pandas as pd

# Optimización (la misma de la Etapa 4)
import scipy
from scipy.optimize import linprog
from scipy.sparse import csr_matrix

# Visualización interactiva
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

SEMILLA = 42

# Paleta fija: el mismo color significa lo mismo en todo el notebook
COLOR_POS = {'Portero': '#7c3aed', 'Defensa': '#2563eb', 'Mediocampista': '#16a34a', 'Delantero': '#dc2626'}
COLOR_EDAD = {'Young': '#0ea5e9', 'Prime': '#f59e0b', 'Veteran': '#475569'}
ORDEN_POS = list(COLOR_POS)
ORDEN_EDAD = list(COLOR_EDAD)
COLOR_GRIS = '#cbd5e1'
COLOR_VENDER, COLOR_RETENER = '#dc2626', '#2563eb'
pio.templates.default = 'plotly_white'
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe


def fmt_eur(v):
    # Formato corto de euros: €1.2M, €350K, €900
    if v is None or pd.isna(v):
        return '-'
    if abs(v) >= 1e6:
        return f'€{v / 1e6:.1f}M'
    if abs(v) >= 1e3:
        return f'€{v / 1e3:.0f}K'
    return f'€{v:.0f}'


def mostrar_tabla(t, formatos=None, titulo=None):
    # Muestra un DataFrame con el mismo estilo en todo el notebook (sin índice, encabezado gris)
    estilo = t.style.hide(axis='index')
    if formatos:
        estilo = estilo.format(formatos, na_rep='-')
    if titulo:
        estilo = estilo.set_caption(titulo)
    estilo = estilo.set_table_styles([
        {'selector': 'caption', 'props': 'caption-side: top; font-weight: bold; font-size: 1.05em; text-align: left; padding: 4px 0 8px;'},
        {'selector': 'th', 'props': 'background: #f1f5f9; color: #0f172a; text-align: left; padding: 4px 10px;'},
        {'selector': 'td', 'props': 'padding: 4px 10px;'},
    ])
    display(estilo)


def tarjetas(items):
    # Indicadores resumidos (KPI) en tarjetas: items = [(título, valor), ...]
    html = ''.join(
        f"<div style='flex:1;min-width:150px;border:1px solid #e2e8f0;border-radius:8px;padding:10px 12px;background:#ffffff'>"
        f"<div style='font-size:12px;color:#64748b'>{k}</div>"
        f"<div style='font-size:20px;font-weight:600;color:#0f172a'>{v}</div></div>"
        for k, v in items)
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin:6px 0 12px'>{html}</div>"))


def mapa_calor(matriz, formato='{:.1f}', escala='YlGnBu', etiquetas=None, **kw):
    # Mapa de calor con el número escrito en cada celda; las celdas sin dato quedan vacías
    texto = matriz.apply(lambda col: col.map(lambda v: '' if pd.isna(v) else formato.format(v)))
    fig = px.imshow(matriz, color_continuous_scale=escala, aspect='auto', labels=etiquetas or {}, **kw)
    fig.update_traces(text=texto.values, texttemplate='%{text}')
    return fig


def aviso(texto, color='#b45309'):
    display(HTML(f"<div style='border-left:4px solid {color};background:#fffbeb;color:#0f172a;padding:8px 12px;margin:6px 0'>{texto}</div>"))


print('Versiones utilizadas:')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | scipy {scipy.__version__}')
print(f'  plotly {plotly.__version__} | ipywidgets {widgets.__version__}')

Versiones utilizadas:
  Python 3.13.18 | pandas 3.0.5 | numpy 2.5.3 | scipy 1.18.1
  plotly 7.1.0 | ipywidgets 8.1.9


### **2. Conexión a Google Drive.**

Si el notebook se ejecuta en Google Colab se intenta montar Drive para leer el CSV de la Etapa 2. Montar Drive es opcional: si no se monta, o si el CSV no está en el Drive de quien abre el notebook, la celda siguiente lo descarga del repositorio del proyecto en GitHub. Así el notebook funciona en cualquier cuenta.

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Ejecutando fuera de Colab: se usará el CSV de la carpeta de trabajo o se descargará del repositorio.')
except Exception as error:
    print(f'No se montó Google Drive ({error}). El CSV se descargará del repositorio del proyecto.')

Ejecutando fuera de Colab: se usará el CSV de la carpeta de trabajo o se descargará del repositorio.


### **3. Carga y homologación del dataset normalizado.**

Es la misma carga de las Etapas 3 y 4: se renombran las variables derivadas (`GrupoEdad`, `TipoPosicion`, `PosicionPrincipal`), se usa el nombre más reciente de cada jugador y `Temporada` se convierte a entero.

In [3]:
FILE_NAME = 'male_players_2015-2024_liga_normalizada.csv'

rutas = [
    f'/content/drive/MyDrive/Etapa_V/dataset/{FILE_NAME}',
    f'/content/drive/MyDrive/Etapa_IV/dataset/{FILE_NAME}',
    FILE_NAME,
]
ruta_csv = next((r for r in rutas if os.path.exists(r)), None)
if ruta_csv is None and os.path.isdir('/content/drive'):
    encontrados = glob.glob(f'/content/drive/**/{FILE_NAME}', recursive=True)
    ruta_csv = encontrados[0] if encontrados else None
if ruta_csv is None:
    # Respaldo: el mismo CSV de la Etapa 2, publicado en el repositorio del proyecto
    import urllib.request
    URL_CSV = f'https://raw.githubusercontent.com/Josee-Arrietaa-31/etapa5-fifa/main/data/{FILE_NAME}'
    print('El CSV no está en Drive ni en la carpeta de trabajo: se descarga del repositorio del proyecto...')
    try:
        urllib.request.urlretrieve(URL_CSV, FILE_NAME)
        ruta_csv = FILE_NAME
    except Exception as error:
        raise FileNotFoundError(f'No se encontró el archivo {FILE_NAME} y no se pudo descargar de {URL_CSV} ({error}).')

df = pd.read_csv(ruta_csv, low_memory=False)
df = df.rename(columns={
    'EdadGrupo': 'GrupoEdad',
    'GrupoPosicion': 'TipoPosicion',
    'posicion_principal': 'PosicionPrincipal',
})
nombre_reciente = df.sort_values('Temporada').groupby('player_id')['short_name'].last()
df['short_name'] = df['player_id'].map(nombre_reciente)
df['Temporada'] = df['Temporada'].astype(int)

print(f'Dataset cargado desde: {ruta_csv}')
print(f'Total de registros: {len(df):,} | Columnas: {df.shape[1]} | Temporadas: {df.Temporada.min()} a {df.Temporada.max()}')

Dataset cargado desde: male_players_2015-2024_liga_normalizada.csv
Total de registros: 175,872 | Columnas: 38 | Temporadas: 2015 a 2024


### **4. Preparación del mercado actual (temporada 2024) para los dashboards.**

Los dashboards del Paso 4 trabajan sobre el mismo "mercado" de la Etapa 4: solo la temporada más reciente, con un registro por jugador, sin jugadores "Sin Club" y sin registros a los que les falte valor, salario o potencial. También se calculan las mismas columnas auxiliares: `valor_por_rating` (euros de valor por punto de `overall`), `valor_estimado` (si el valor tiene decimales, fue estimado en la Etapa 2) y `cat` (la posición que usan los modelos de equipo).

In [4]:
faltantes = df[['value_eur', 'wage_eur', 'potential']].isna().any(axis=1)
df_val = df.loc[~faltantes].copy()

TEMPORADA_MERCADO = int(df_val['Temporada'].max())
snap = df_val[df_val['Temporada'] == TEMPORADA_MERCADO].copy()
assert snap['player_id'].is_unique, 'Hay jugadores repetidos en la temporada de mercado'
snap = snap[snap['club_name'] != 'Sin Club'].copy()
snap['valor_por_rating'] = snap['value_eur'] / snap['overall']
snap[['age', 'overall']] = snap[['age', 'overall']].astype(int)
snap['valor_estimado'] = np.where(snap['value_eur'] % 1 != 0, 'Sí', 'No')
snap['cat'] = snap['TipoPosicion']
snap = snap.reset_index(drop=True)

print(f'Registros con valor, salario y potencial: {len(df_val):,} de {len(df):,}')
print(f'Mercado actual (temporada {TEMPORADA_MERCADO}): {len(snap):,} jugadores únicos')
print('Jugadores por posición:', snap['cat'].value_counts().reindex(ORDEN_POS).to_dict())

Registros con valor, salario y potencial: 174,138 de 175,872
Mercado actual (temporada 2024): 14,056 jugadores únicos
Jugadores por posición: {'Portero': 1587, 'Defensa': 4802, 'Mediocampista': 5092, 'Delantero': 2575}


**Interpretación de los resultados.**

La carga reproduce la base de la Etapa 4: los registros sin valor de mercado, salario o potencial (todos de 2024) se descartan, y el mercado actual queda formado por los jugadores de la temporada 2024, uno por registro, repartidos en las cuatro posiciones que usan los modelos de equipo. Como la preparación es idéntica, los dashboards pueden compararse directamente con los resultados de la Etapa 4.

## **Paso 2: Inventario de Preguntas y Elección de la Representación**

> *Identificar las preguntas que se respondieron en cada una de las secciones. Para cada respuesta debe elegir cómo representarla: tablas o plots.*

Se recorrieron los cuatro notebooks anteriores y se listó cada pregunta de negocio que se respondió, con su respuesta resumida. Para cada una se eligió una representación aplicando la regla definida al inicio, y se indica en qué parte del Paso 3 se construyó.

#### **Contenido de la sección:**
1. Inventario completo de preguntas y representación elegida.
2. Resumen del inventario.

### **1. Inventario completo de preguntas y representación elegida.**

In [5]:
I = []
def pregunta(etapa, seccion, preg, respuesta, representacion, visual, porque, donde):
    I.append({'Etapa': etapa, 'Sección': seccion, 'Pregunta': preg, 'Respuesta (resumen)': respuesta,
              'Representación': representacion, 'Visual': visual, 'Por qué': porque, 'Dónde (Paso 3)': donde})

# ---------------- Etapa 1: Preprocesado y Limpieza ----------------
pregunta('Etapa 1', 'Paso 3', '¿Qué fuente usar para 2015-2023: legacy o la última actualización?',
         'La última actualización de cada edición: incluye transferencias y ajustes de toda la temporada.',
         'Tabla', 'Tabla de decisión', 'Es una decisión con pocas cifras exactas, no una comparación visual.', '3.1 · Tabla 1.1')
pregunta('Etapa 1', 'Pasos 5-10', '¿Qué calidad tiene el dataset final (registros, duplicados, nulos)?',
         'Un registro por jugador y temporada, sin duplicados; los faltantes se concentran en 2024.',
         'Tabla', 'Informe de calidad', 'El lector necesita los valores exactos de cada indicador.', '3.1 · Tabla 1.1')
pregunta('Etapa 1', 'Paso 8', '¿Qué features derivados se crearon y para qué?',
         'EdadGrupo, GrupoPosicion, BrechaPotencial, IMC, Ratio_Valor_Salario_Anual y Rating_Promedio_Atributos.',
         'Tabla', 'Diccionario de features', 'Son definiciones y fórmulas, no magnitudes.', '3.1 · Tabla 1.2')
pregunta('Etapa 1', 'Paso 9', '¿Cuántos registros marcan los features de seguridad y en qué temporadas?',
         'Tres banderas (edad sospechosa, habilidades bajas, valor-salario inconsistente) con umbrales por percentiles.',
         'Gráfico', 'Barras por temporada + línea de %', 'Compara una cantidad entre temporadas.', '3.1 · Fig. 1.1')

# ---------------- Etapa 2: Modelado Dimensional ----------------
pregunta('Etapa 2', 'Consulta 1', 'Top 10 clubes por promedio de valor de mercado por temporada.',
         'Barcelona, Bayern, Real Madrid, PSG y Manchester City están en el top 10 de las 10 temporadas.',
         'Gráfico', 'Mapa de calor club × temporada (puesto)', 'Muestra la permanencia en el top a lo largo del tiempo; una tabla de 100 filas no se lee.', '3.2 · Fig. 2.1')
pregunta('Etapa 2', 'Consulta 2', 'Valor total de mercado por región y por liga.',
         'Europa concentra el 84.7 % del valor total; la Premier League es la liga de mayor valor.',
         'Gráfico', 'Treemap región → liga', 'Es una jerarquía y lo importante es la proporción del total.', '3.2 · Fig. 2.2')
pregunta('Etapa 2', 'Consulta 3', 'Rendimiento (overall) medio por grupo de edad y por posición.',
         'Veteran > Prime > Young en todas las posiciones; el máximo es CF Veteran (75.2).',
         'Gráfico', 'Mapa de calor posición × grupo de edad', 'Cruza dos categorías con una medida.', '3.2 · Fig. 2.3')
pregunta('Etapa 2', 'Consulta 4', 'Análisis de márgenes (valor − salario) por club y por liga.',
         'Real Madrid, Manchester City y Barcelona tienen los mayores márgenes acumulados.',
         'Tabla', 'Top 10 clubes y top 10 ligas', 'Interesan los montos exactos de pocos clubes y ligas.', '3.2 · Tablas 2.1 y 2.2')
pregunta('Etapa 2', 'Consulta 5', 'Drill-down Club → Liga → Región con valor y salario promedio.',
         'Europa lidera en valor y salario promedio; la Premier League encabeza entre las ligas.',
         'Gráfico', 'Sunburst interactivo', 'El drill-down se hace con clics, que es exactamente la operación pedida.', '3.2 · Fig. 2.4')
pregunta('Etapa 2', 'Paso 6', '¿Cuánto mejora el tiempo de la consulta con un índice en la FK?',
         'De 7.57 ms a 7.05 ms (6.9 % menos).',
         'Tabla', 'Tabla de tiempos', 'Son tres números exactos.', '3.2 · Tabla 2.3')
pregunta('Etapa 2', 'Paso 4.2', '¿El pipeline carga temporada por temporada sin duplicar registros?', 'Sí: 10 lotes de 15,623 a 19,238 registros, 175,872 en total y 0 duplicados (player_id, Temporada).', 'Gráfico', 'Barras por temporada + línea acumulada', 'Compara la carga entre temporadas y muestra cómo crece la tabla.', '3.2 · Fig. 2.5')

# ---------------- Etapa 3: Análisis de Datos 1 ----------------
pregunta('Etapa 3', 'Univariante', 'Medidas de tendencia central y dispersión de 7 variables numéricas.',
         'Edad, overall y potencial son casi simétricos; valor y salario tienen una fuerte asimetría positiva.',
         'Tabla', 'Tabla de estadísticos', 'Son muchas medidas exactas por variable.', '3.3 · Tabla 3.1')
pregunta('Etapa 3', 'Univariante', '¿Cómo se distribuye cada variable? (histograma y boxplot)',
         'Ver el explorador interactivo: cada variable con su media, mediana y cuartiles.',
         'Gráfico', 'Explorador de distribuciones (widget)', 'Una distribución se entiende viéndola.', '3.3 · Fig. 3.1')
pregunta('Etapa 3', 'Preg. 1', '¿Qué club tiene más jugadores en el dataset?',
         'Arsenal, con 342 registros jugador-temporada.', 'Gráfico', 'Barras (ranking interactivo)', 'Comparación entre pocas categorías.', '3.3 · Fig. 3.5')
pregunta('Etapa 3', 'Preg. 2', '¿Qué nacionalidad tiene mejor rating promedio?',
         'Brazil (70.84), con menos de un punto de diferencia sobre las siguientes.', 'Gráfico', 'Barras (ranking interactivo)', 'Comparación entre pocas categorías.', '3.3 · Fig. 3.5')
pregunta('Etapa 3', 'Preg. 3-6', '¿Edad más común, distribución por GrupoEdad, sesgo y anomalías de edad?',
         'Moda de 22 años; Prime 57.1 %, Veteran 25.0 %, Young 18.0 %; leve sesgo a la derecha; 330 edades fuera de 17-40.',
         'Gráfico', 'Explorador de distribuciones', 'Son preguntas sobre la forma de una distribución.', '3.3 · Fig. 3.1')
pregunta('Etapa 3', 'Preg. 7', '¿Cómo se distribuye el overall por GrupoEdad?',
         'La mediana sube con la edad: 59 (Young), 66 (Prime) y 69 (Veteran).', 'Gráfico', 'Explorador (overall por GrupoEdad)', 'Compara distribuciones entre grupos.', '3.3 · Fig. 3.1')
pregunta('Etapa 3', 'Bivariante', '¿El potencial correlaciona más con el valor o con el salario?',
         'Algo más con el valor (0.53) que con el salario (0.50).', 'Gráfico', 'Mapa de calor de correlaciones', 'Muestra todas las relaciones a la vez.', '3.3 · Fig. 3.2')
pregunta('Etapa 3', 'Bivariante', '¿A mayor edad, mayor salario? ¿Qué posición cobra más?',
         'El salario medio sube con la edad; CAM, ST y CM tienen los promedios más altos, con medianas casi iguales.',
         'Gráfico', 'Explorador (salario por grupo)', 'Compara distribuciones entre grupos.', '3.3 · Fig. 3.1')
pregunta('Etapa 3', 'Multivariante', '¿Combinación de edad y posición con mayor valor de mercado?',
         'Delanteros Prime y Veteran; el menor valor es el del portero joven.', 'Gráfico', 'Mapa de calor posición × edad', 'Cruza dos categorías con una medida.', '3.3 · Fig. 3.4')
pregunta('Etapa 3', 'Multivariante', '¿Cómo se relacionan el valor y el overall según la edad? ¿Hay clusters?',
         'El valor crece de forma exponencial con el overall; el cluster natural lo forma la edad, no la posición.',
         'Gráfico', 'Dispersión con escala logarítmica', 'Relación entre dos variables numéricas y una categórica.', '3.3 · Fig. 3.3')
pregunta('Etapa 3', 'Anomalías', 'Top 5 de BrechaPotencial.',
         'Callum Burton lidera con 28 puntos (overall 50, potencial 78).', 'Tabla', 'Tabla de jugadores', 'Se necesita identificar a cada jugador.', '3.3 · Tabla 3.2')
pregunta('Etapa 3', 'Anomalías', 'Top 5 de subvalorados (overall / salario).',
         'Jugadores con overall medio y salario mínimo.', 'Tabla', 'Tabla de jugadores', 'Se necesita identificar a cada jugador.', '3.3 · Tabla 3.3')
pregunta('Etapa 3', 'Anomalías', '¿Hay jugadores con salario alto pero rating bajo?',
         'Sí, unos pocos (por ejemplo, Pape Souaré: €33,143 semanales con overall 61).', 'Tabla', 'Tabla de jugadores', 'Son pocos casos que hay que revisar uno por uno.', '3.3 · Tabla 3.4')
pregunta('Etapa 3', 'Global', 'Top 10 por valor, por potencial, jóvenes con rating > 80, clubes y nacionalidades.',
         'Mbappé y Haaland lideran en valor; Bayern, Juventus y Barcelona en rating; Spain en valor total.',
         'Tabla + gráfico', 'Ranking interactivo (tabla y barras)', 'Los nombres requieren tabla; la barra muestra la distancia entre puestos.', '3.3 · Fig. 3.5')
pregunta('Etapa 3', 'Inversiones', 'Mejores gangas: alto overall y bajo valor.',
         'Jugadores veteranos de overall 70+ con valor en el 25 % inferior.', 'Tabla + gráfico', 'Ranking interactivo', 'Lista de jugadores a identificar.', '3.3 · Fig. 3.5')
pregunta('Etapa 3', 'Univariante', 'Top 10 nacionalidades y clubes; distribución por TipoPosición, GrupoEdad y pie dominante.', 'England (16,235) y Arsenal (342) encabezan; Mediocampista 37.0 %, Prime 57.1 % y diestros 76.5 %.', 'Gráfico', 'Barras (cinco paneles)', 'Compara frecuencias entre pocas categorías.', '3.3 · Fig. 3.6')
pregunta('Etapa 3', 'Bivariante · Preg. 4', '¿Diferencia en overall entre diestros y zurdos?', 'No hay una diferencia importante: la mediana es 66 en los dos grupos.', 'Gráfico', 'Boxplot por pie dominante', 'Compara dos distribuciones.', '3.3 · Fig. 3.7')
pregunta('Etapa 3', 'Bivariante · Preg. 5', '¿Liga con promedio de value más alto?', 'Premier League (€9.7M), seguida de La Liga (€8.8M), Serie A y Bundesliga.', 'Gráfico', 'Barras horizontales coloreadas por región', 'Ranking de pocas categorías.', '3.3 · Fig. 3.8')
pregunta('Etapa 3', 'Multivariante · Preg. 1', '¿Jugadores zurdos por posición?', 'Los defensores tienen la mayor proporción de zurdos (33.1 %) y los porteros la menor (12.4 %).', 'Gráfico', 'Barras de % de zurdos', 'Compara una proporción entre cuatro grupos.', '3.3 · Fig. 3.7')
pregunta('Etapa 3', 'Multivariante · Preg. 2', '¿Jóvenes mejor pagados por posición?', 'Dembélé (ofensivos, €150,000), Pedri (€115,000), Luke Shaw (€90,000) y Donnarumma (€34,000).', 'Tabla', 'Tabla de jugadores', 'Se necesita identificar a cada jugador.', '3.3 · Tabla 3.6')
pregunta('Etapa 3', 'Anomalías', 'Z-score en edad y salario e IQR en BrechaPotencial: ¿qué registros son anómalos?', '227 por edad (40 años o más), 3,447 por salario (más de €75,347) y 816 por brecha (más de 20 puntos).', 'Tabla', 'Resumen por método con los casos extremos', 'Son límites y conteos exactos y nombres de jugadores.', '3.3 · Tabla 3.7')
pregunta('Etapa 3', 'Anomalías', 'Análisis manual de 3 registros: ¿estrella, gema o error de datos?', 'Cristiano Ronaldo (estrella), Callum Burton (gema) y F. Lampard (posible error); los tres se conservan.', 'Tabla', 'Tabla con decisión y acción', 'Cada fila es un caso con su decisión.', '3.3 · Tabla 3.8')
pregunta('Etapa 3', 'Anomalías · Preg. 3', 'Clubes con mayor potencial promedio.', 'FC Barcelona (84.15), Real Madrid CF (84.08) y FC Bayern München (84.07), prácticamente empatados.', 'Gráfico', 'Barras horizontales', 'Ranking de pocas categorías.', '3.3 · Fig. 3.9')
pregunta('Etapa 3', 'Global', 'Grupo de edad con mayor potencial.', 'Young (72.3), frente a Prime (71.2) y Veteran (69.3).', 'Gráfico', 'Barras', 'Compara tres grupos.', '3.3 · Fig. 3.9')
pregunta('Etapa 3', 'Global', 'Nacionalidad con salario medio más alto.', 'Gabon (€22,408) entre las nacionalidades con al menos 100 registros; le siguen Algeria y Brazil.', 'Gráfico', 'Barras horizontales', 'Ranking de pocas categorías.', '3.3 · Fig. 3.9')
pregunta('Etapa 3', 'Global', 'Distribución por posición.', 'Entre los titulares: mediocampistas 37 %, defensores 36 %, ofensivos 17 % y porteros 10 %.', 'Gráfico', 'Barras', 'Compara frecuencias entre cuatro grupos.', '3.3 · Fig. 3.10')
pregunta('Etapa 3', 'Reclutamiento', 'Jóvenes menores de 21 años: brecha de potencial frente a rating.', 'Felipe Valencia y Peder Hoel Lervik lideran con 25 puntos de brecha y overall de 50 y 56.', 'Tabla', 'Tabla de jugadores', 'Se necesita identificar a cada jugador.', '3.3 · Tabla 3.9')
pregunta('Etapa 3', 'Inversiones', 'Club con mejor ratio value/wage.', 'Dynamo Kyiv (5,976), explicado por salarios registrados muy bajos.', 'Tabla', 'Top 5 de clubes', 'Interesan los valores exactos de pocos clubes.', '3.3 · Tabla 3.10')
pregunta('Etapa 3', 'Mercado', 'Nacionalidades predominantes entre los jugadores de alto valor.', 'Spain (2,146 registros), Brazil (1,544) y France (1,487) en el 10 % de mayor valor.', 'Gráfico', 'Barras horizontales', 'Ranking de pocas categorías.', '3.3 · Fig. 3.9')
pregunta('Etapa 3', 'Mercado', 'Attackers vs. defenders.', 'Salario medio de €12,546 frente a €10,104, pero la mediana es €4,000 en ambos grupos.', 'Gráfico', 'Barras agrupadas (media y mediana)', 'Compara dos medidas entre pocos grupos.', '3.3 · Fig. 3.10')
pregunta('Etapa 3', 'Profiling', 'Perfil del centrocampista de élite.', '294 registros: 27.8 años, overall 86.5, €54.0M y €173,703 semanales; destaca en passing y dribbling.', 'Tabla', 'Perfil frente a otros grupos', 'Se comparan muchas medidas para tres filas.', '3.3 · Tabla 3.11')
pregunta('Etapa 3', 'Profiling', 'Influencia del pie dominante.', 'Mínima: los zurdos son el 23.5 % y tienen la misma mediana de overall y de salario que los diestros.', 'Tabla', 'Perfil por pie dominante', 'Pocas filas con varias medidas.', '3.3 · Tabla 3.5')

# ---------------- Etapa 4: Análisis Predictivo y Prescriptivo ----------------
pregunta('Etapa 4', 'Predictivo 1', '¿Qué variables predicen el valor de mercado y con qué error (RMSE)?',
         'Regresión lineal sobre log(valor): R² 0.904, RMSE 0.4164; top 5: overall, age, potential, defending, shooting.',
         'Tabla + gráfico', 'Métricas + real vs predicho + coeficientes', 'La métrica se lee exacta; el gráfico muestra dónde acierta y dónde falla.', '3.4 · Tabla 4.1 · Figs. 4.2 y 4.3')
pregunta('Etapa 4', 'Predictivo 2', '¿Se puede predecir si un jugador tendrá potencial > 85?',
         'Random Forest: F1 0.75 y PR-AUC 0.82; la regresión logística tiene recall 0.97 pero precisión 0.20. Accuracy de 0.99 (Random Forest) y 0.95 (logística), poco informativa por el desbalance de clases.',
         'Tabla + gráfico', 'Métricas + matrices de confusión + ROC', 'Lo pide el enunciado; la matriz muestra el tipo de error de cada modelo.', '3.4 · Tabla 4.1 · Fig. 4.4')
pregunta('Etapa 4', 'Predictivo 3', '¿Qué variables predicen el salario?',
         'El overall domina (R² 0.60); con región, posición y temporada el R² sube a 0.70.',
         'Tabla', 'Tabla de métricas de modelos', 'Las métricas se leen como valores exactos.', '3.4 · Tabla 4.1')
pregunta('Etapa 4', 'Predictivo 4', '¿Qué tipos de jugador hay (K-Means)?',
         'Alto valor 21.4 %, veteranos consolidados 28.9 %, promesas 27.6 %, jugadores de base 22.1 %.',
         'Gráfico', 'Barras de tamaño + dispersión PCA', 'Compara la proporción de los grupos y muestra cómo se separan.', '3.4 · Figs. 4.1 y 4.5')
pregunta('Etapa 4', 'Prescriptivo 1', 'Mercado de transferencias: 5 jóvenes de potencial > 85 con €100M.',
         'Moukoko, Miretti, Cherki, Nusa y António Silva: €41.0M y potencial promedio de 88.0.',
         'Dashboard', 'Dashboard interactivo', 'El usuario puede cambiar presupuesto, edad, potencial y criterio.', 'Paso 4 · Dashboard 1')
pregunta('Etapa 4', 'Prescriptivo 2', 'Construcción de equipo 1-4-3-3 con menos de €150M (greedy vs linprog).',
         'linprog: rating 932 por €145.6M; greedy: 832 por €150.0M.',
         'Dashboard', 'Dashboard interactivo', 'El usuario puede cambiar presupuesto, formación y método.', 'Paso 4 · Dashboard 2')
pregunta('Etapa 4', 'Prescriptivo 3', 'Vender o no a un jugador de 28 años, overall 82, potencial 83 y salario alto.',
         'Vender si la oferta es de unos €16M o más; retener si es menor.',
         'Dashboard', 'Dashboard interactivo', 'El usuario puede cambiar los datos del jugador y la oferta.', 'Paso 4 · Dashboard 3')
pregunta('Etapa 4', 'Prescriptivo 4', 'Contratación: top 3 más eficientes por posición.',
         'Nicolaisen (DEF), Abdul Samed (MED) y Merentiel (DEL), entre €1.1M y €1.2M.',
         'Tabla', 'Tablas top 3 por posición', 'Lista de jugadores a identificar; no tiene entradas que el usuario cambie.', '3.4 · Tablas 4.2 y 4.3')
pregunta('Etapa 4', 'BI', 'Cubos por club, región/liga, edad y región × edad × temporada.',
         'El valor mediano casi se triplicó entre 2015 y 2024 sin que subiera el nivel; Europa concentra la oferta joven.',
         'Gráfico', 'Línea (roll-up) y mapa de calor (slice)', 'Tendencia en el tiempo y cruce de dos categorías.', '3.4 · Figs. 4.6 y 4.7')

inventario = pd.DataFrame(I)
print(f'Preguntas inventariadas: {len(inventario)}')
mostrar_tabla(inventario, titulo='Inventario de preguntas respondidas en las Etapas 1 a 4')

Preguntas inventariadas: 52


Etapa,Sección,Pregunta,Respuesta (resumen),Representación,Visual,Por qué,Dónde (Paso 3)
Etapa 1,Paso 3,¿Qué fuente usar para 2015-2023: legacy o la última actualización?,La última actualización de cada edición: incluye transferencias y ajustes de toda la temporada.,Tabla,Tabla de decisión,"Es una decisión con pocas cifras exactas, no una comparación visual.",3.1 · Tabla 1.1
Etapa 1,Pasos 5-10,"¿Qué calidad tiene el dataset final (registros, duplicados, nulos)?","Un registro por jugador y temporada, sin duplicados; los faltantes se concentran en 2024.",Tabla,Informe de calidad,El lector necesita los valores exactos de cada indicador.,3.1 · Tabla 1.1
Etapa 1,Paso 8,¿Qué features derivados se crearon y para qué?,"EdadGrupo, GrupoPosicion, BrechaPotencial, IMC, Ratio_Valor_Salario_Anual y Rating_Promedio_Atributos.",Tabla,Diccionario de features,"Son definiciones y fórmulas, no magnitudes.",3.1 · Tabla 1.2
Etapa 1,Paso 9,¿Cuántos registros marcan los features de seguridad y en qué temporadas?,"Tres banderas (edad sospechosa, habilidades bajas, valor-salario inconsistente) con umbrales por percentiles.",Gráfico,Barras por temporada + línea de %,Compara una cantidad entre temporadas.,3.1 · Fig. 1.1
Etapa 2,Consulta 1,Top 10 clubes por promedio de valor de mercado por temporada.,"Barcelona, Bayern, Real Madrid, PSG y Manchester City están en el top 10 de las 10 temporadas.",Gráfico,Mapa de calor club × temporada (puesto),Muestra la permanencia en el top a lo largo del tiempo; una tabla de 100 filas no se lee.,3.2 · Fig. 2.1
Etapa 2,Consulta 2,Valor total de mercado por región y por liga.,Europa concentra el 84.7 % del valor total; la Premier League es la liga de mayor valor.,Gráfico,Treemap región → liga,Es una jerarquía y lo importante es la proporción del total.,3.2 · Fig. 2.2
Etapa 2,Consulta 3,Rendimiento (overall) medio por grupo de edad y por posición.,Veteran > Prime > Young en todas las posiciones; el máximo es CF Veteran (75.2).,Gráfico,Mapa de calor posición × grupo de edad,Cruza dos categorías con una medida.,3.2 · Fig. 2.3
Etapa 2,Consulta 4,Análisis de márgenes (valor − salario) por club y por liga.,"Real Madrid, Manchester City y Barcelona tienen los mayores márgenes acumulados.",Tabla,Top 10 clubes y top 10 ligas,Interesan los montos exactos de pocos clubes y ligas.,3.2 · Tablas 2.1 y 2.2
Etapa 2,Consulta 5,Drill-down Club → Liga → Región con valor y salario promedio.,Europa lidera en valor y salario promedio; la Premier League encabeza entre las ligas.,Gráfico,Sunburst interactivo,"El drill-down se hace con clics, que es exactamente la operación pedida.",3.2 · Fig. 2.4
Etapa 2,Paso 6,¿Cuánto mejora el tiempo de la consulta con un índice en la FK?,De 7.57 ms a 7.05 ms (6.9 % menos).,Tabla,Tabla de tiempos,Son tres números exactos.,3.2 · Tabla 2.3


### **2. Resumen del inventario.**

In [6]:
resumen = pd.crosstab(inventario['Etapa'], inventario['Representación'])
print(resumen.to_string())

largo = resumen.reset_index().melt(id_vars='Etapa', var_name='Representación', value_name='Preguntas')
fig = px.bar(largo, x='Etapa', y='Preguntas', color='Representación', text='Preguntas',
             color_discrete_map={'Tabla': '#64748b', 'Gráfico': '#2563eb', 'Tabla + gráfico': '#0ea5e9', 'Dashboard': '#dc2626'},
             title='Representación elegida para cada pregunta, por etapa')
fig.update_layout(height=380, legend_title_text='')
fig.show()

Representación  Dashboard  Gráfico  Tabla  Tabla + gráfico
Etapa                                                     
Etapa 1                 0        1      3                0
Etapa 2                 0        5      2                0
Etapa 3                 0       19     11                2
Etapa 4                 3        2      2                2


**Interpretación de los resultados.**

El inventario reúne las preguntas de negocio de las cuatro etapas anteriores, cada una con su respuesta y la representación elegida. El gráfico muestra cómo se reparte esa elección por etapa.

La Etapa 1 se representa sobre todo con tablas, porque sus respuestas son decisiones y definiciones (qué fuente usar, qué features se crearon) y no comparaciones. La Etapa 3 usa sobre todo gráficos, porque sus preguntas tratan sobre distribuciones, relaciones y rankings. En la Etapa 4, las métricas de los modelos se dejan en tabla porque se leen como valores exactos, y los tres primeros experimentos prescriptivos pasan a dashboards, como pide el enunciado.

En conclusión, la elección no es arbitraria: se usa tabla cuando el lector necesita nombres o valores exactos, y gráfico cuando necesita ver una comparación, una forma o una tendencia.

## **Paso 3: Representación de las Respuestas por Etapa**

En este paso se construye la representación elegida para cada pregunta del inventario. Los gráficos son interactivos: al pasar el cursor se ve el valor exacto, la leyenda permite ocultar series y la barra superior permite hacer zoom o descargar la imagen.

#### **Contenido de la sección:**
1. Etapa 1: calidad del dataset.
2. Etapa 2: consultas del modelo dimensional.
3. Etapa 3: análisis descriptivo, de diagnóstico y exploratorio.
4. Etapa 4: modelos predictivos y cubos de BI.

### **1. Etapa 1: calidad del dataset.**

**Tabla 1.1** resume la calidad del dataset final, **Tabla 1.2** describe los features derivados y **Fig. 1.1** muestra cuántos registros marcan los features de seguridad en cada temporada.

In [7]:
# Tabla 1.1 — Informe de calidad del dataset final
dups = int(df.duplicated(subset=['player_id', 'Temporada']).sum())
d24 = df[df['Temporada'] == df['Temporada'].max()]
calidad = pd.DataFrame([
    ('Fuente 2015-2023', 'Última actualización de cada edición (no legacy)', 'Incluye transferencias y ajustes de la temporada'),
    ('Registros (jugador-temporada)', f'{len(df):,}', 'Un registro por jugador y temporada'),
    ('Jugadores únicos', f"{df['player_id'].nunique():,}", ''),
    ('Temporadas', f"{df['Temporada'].min()} a {df['Temporada'].max()}", f"{df['Temporada'].nunique()} temporadas"),
    ('Columnas', f'{df.shape[1]}', 'Incluye 6 features derivados y 4 de seguridad'),
    ('Duplicados (player_id, Temporada)', f'{dups:,}', 'Debe ser 0'),
    ('Registros sin valor, salario o potencial', f'{int(faltantes.sum()):,}', 'Jugadores de 2024 sin historial; no se imputaron'),
    ('Valores de mercado estimados en 2024', f"{(d24['value_eur'] % 1 != 0).mean() * 100:.1f}%", 'Tienen decimales: se rellenaron desde el historial del jugador'),
    ("Registros 'Sin Club'", f"{(df['club_name'] == 'Sin Club').sum():,}", 'Agentes libres, no datos faltantes'),
], columns=['Indicador', 'Valor', 'Comentario'])
mostrar_tabla(calidad, titulo='Tabla 1.1 · Informe de calidad del dataset final 2015-2024')

# Tabla 1.2 — Features derivados (Etapa 1, Paso 8)
features = pd.DataFrame([
    ('EdadGrupo → GrupoEdad', 'Young (< 21), Prime (21-28), Veteran (> 28)', 'Comparar etapas de la carrera'),
    ('GrupoPosicion → TipoPosicion', 'Posición principal agrupada en Portero, Defensa, Mediocampista y Delantero', 'Armar equipos y comparar posiciones'),
    ('BrechaPotencial', 'potential − overall', 'Medir el margen de crecimiento'),
    ('IMC', 'weight_kg / (height_cm / 100)²', 'Perfil físico'),
    ('Ratio_Valor_Salario_Anual', 'value_eur / (wage_eur × 52)', 'Cuánto valor respalda cada euro de salario anual'),
    ('Rating_Promedio_Atributos', 'Promedio de pace, shooting, passing, dribbling, defending y physic', 'Nivel técnico independiente del overall'),
], columns=['Feature', 'Definición', 'Para qué se usa'])
mostrar_tabla(features, titulo='Tabla 1.2 · Features derivados creados en la Etapa 1')

Indicador,Valor,Comentario
Fuente 2015-2023,Última actualización de cada edición (no legacy),Incluye transferencias y ajustes de la temporada
Registros (jugador-temporada),"175,872",Un registro por jugador y temporada
Jugadores únicos,"51,052",
Temporadas,2015 a 2024,10 temporadas
Columnas,38,Incluye 6 features derivados y 4 de seguridad
"Duplicados (player_id, Temporada)",0,Debe ser 0
"Registros sin valor, salario o potencial","1,734",Jugadores de 2024 sin historial; no se imputaron
Valores de mercado estimados en 2024,39.1%,Tienen decimales: se rellenaron desde el historial del jugador
Registros 'Sin Club',"1,718","Agentes libres, no datos faltantes"


Feature,Definición,Para qué se usa
EdadGrupo → GrupoEdad,"Young (< 21), Prime (21-28), Veteran (> 28)",Comparar etapas de la carrera
GrupoPosicion → TipoPosicion,"Posición principal agrupada en Portero, Defensa, Mediocampista y Delantero",Armar equipos y comparar posiciones
BrechaPotencial,potential − overall,Medir el margen de crecimiento
IMC,weight_kg / (height_cm / 100)²,Perfil físico
Ratio_Valor_Salario_Anual,value_eur / (wage_eur × 52),Cuánto valor respalda cada euro de salario anual
Rating_Promedio_Atributos,"Promedio de pace, shooting, passing, dribbling, defending y physic",Nivel técnico independiente del overall


In [8]:
# Fig. 1.1 — Registros por temporada y porcentaje marcado por los features de seguridad
def a_bool(s):
    # Las banderas pueden venir como bool, 0/1 o texto según cómo se exportó el CSV
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'si', 'sí', 'yes'])

banderas = {'flag_edad_sospechosa': 'Edad sospechosa', 'flag_habilidades_bajas': 'Habilidades bajas',
            'flag_valor_salario_inconsistente': 'Valor-salario inconsistente'}
marcas = pd.DataFrame({nombre: a_bool(df[col]) for col, nombre in banderas.items() if col in df.columns})
marcas['Temporada'] = df['Temporada']
por_temp = marcas.groupby('Temporada').sum()
total_temp = df.groupby('Temporada').size()
alguna = marcas.drop(columns='Temporada').any(axis=1).groupby(df['Temporada']).mean() * 100

fig = make_subplots(specs=[[{'secondary_y': True}]])
for i, col in enumerate(por_temp.columns):
    fig.add_bar(x=por_temp.index, y=por_temp[col], name=col, marker_color=['#f59e0b', '#7c3aed', '#dc2626'][i])
fig.add_scatter(x=alguna.index, y=alguna.values, name='% con alguna bandera', mode='lines+markers',
                line=dict(color='#0f172a', width=2), secondary_y=True)
fig.update_layout(barmode='stack', height=420, title='Fig. 1.1 · Registros marcados por los features de seguridad, por temporada',
                  legend=dict(orientation='h', y=-0.2))
fig.update_yaxes(title_text='Registros marcados', secondary_y=False)
fig.update_yaxes(title_text='% de la temporada', secondary_y=True, rangemode='tozero')
fig.update_xaxes(title_text='Temporada', dtick=1)
fig.show()

print(f'Registros por temporada: {total_temp.to_dict()}')
print(f"Registros con al menos una bandera: {int(marcas.drop(columns='Temporada').any(axis=1).sum()):,} ({marcas.drop(columns='Temporada').any(axis=1).mean() * 100:.1f}%)")

Registros por temporada: {2015: 16155, 2016: 15623, 2017: 17596, 2018: 17954, 2019: 18085, 2020: 18483, 2021: 18943, 2022: 19238, 2023: 18005, 2024: 15790}
Registros con al menos una bandera: 4,147 (2.4%)


**Interpretación de los resultados.**

La Tabla 1.1 resume en una sola vista las decisiones de limpieza de la Etapa 1: la fuente elegida, el tamaño del dataset, la ausencia de duplicados y los faltantes que se dejaron sin imputar, todos de jugadores de 2024 sin historial. La Tabla 1.2 recuerda qué mide cada feature derivado, porque varios se usan en los gráficos siguientes. La Fig. 1.1 muestra que las banderas de seguridad marcan una fracción pequeña de cada temporada, con un porcentaje parecido de un año a otro.

Las banderas se calcularon con percentiles del propio dataset, por lo que es esperable que marquen una proporción estable por temporada; un salto en una temporada indicaría un problema de captura en esa edición.

En conclusión, el dataset que alimenta las etapas siguientes es consistente, y los registros dudosos están identificados pero no se eliminaron, de modo que cada análisis puede decidir si los usa.

### **2. Etapa 2: consultas del modelo dimensional.**

Se representan las cinco consultas de negocio del modelo estrella, la medición de tiempos y el pipeline de carga por temporada (**Fig. 2.5**). Los cálculos se hacen con pandas sobre el mismo CSV, con las mismas definiciones que las consultas SQL de la Etapa 2.

In [9]:
# Fig. 2.1 — Consulta 1: top 10 clubes por valor promedio, por temporada
base = df[(df['club_name'] != 'Sin Club')].dropna(subset=['value_eur', 'wage_eur'])
prom = base.groupby(['Temporada', 'club_name'])['value_eur'].mean().reset_index()
prom['Puesto'] = prom.groupby('Temporada')['value_eur'].rank(ascending=False, method='first')
top10 = prom[prom['Puesto'] <= 10]
apariciones = (top10.groupby('club_name')
               .agg(Temporadas_en_top=('Puesto', 'size'), Puesto_promedio=('Puesto', 'mean'))
               .sort_values(['Temporadas_en_top', 'Puesto_promedio'], ascending=[False, True]))
matriz = top10.pivot(index='club_name', columns='Temporada', values='Puesto').reindex(apariciones.index)

fig = mapa_calor(matriz, '{:.0f}', 'Blues_r', dict(x='Temporada', y='', color='Puesto'))
fig.update_layout(title='Fig. 2.1 · Puesto en el top 10 de valor promedio por temporada (celda vacía = fuera del top 10)',
                  height=max(420, 24 * len(matriz) + 140))
fig.update_xaxes(dtick=1, side='top')
fig.show()

siempre = apariciones[apariciones['Temporadas_en_top'] == df['Temporada'].nunique()].index.tolist()
print(f'Clubes que aparecen al menos una vez en el top 10: {len(matriz)}')
print(f'Clubes presentes en todas las temporadas: {siempre}')

Clubes que aparecen al menos una vez en el top 10: 17
Clubes presentes en todas las temporadas: ['FC Bayern München', 'Real Madrid CF', 'FC Barcelona', 'Paris Saint-Germain', 'Manchester City']


In [10]:
# Fig. 2.2 — Consulta 2: valor total de mercado por región y liga (treemap)
reg = df.dropna(subset=['value_eur']).assign(
    league_region=lambda d: d['league_region'].fillna('Sin Clasificar'),
    league_name=lambda d: d['league_name'].fillna('Sin Liga'))
valor_liga = reg.groupby(['league_region', 'league_name'], as_index=False)['value_eur'].sum()
valor_liga['Valor (M€)'] = valor_liga['value_eur'] / 1e6

fig = px.treemap(valor_liga, path=[px.Constant('Total'), 'league_region', 'league_name'], values='Valor (M€)',
                 color='league_region', color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_traces(texttemplate='<b>%{label}</b><br>€%{value:,.0f}M<br>%{percentRoot:.1%} del total',
                  hovertemplate='%{label}<br>€%{value:,.0f}M<br>%{percentRoot:.1%} del total<extra></extra>')
fig.update_layout(title='Fig. 2.2 · Valor total de mercado por región y liga (clic en una región para ampliar)', height=560,
                  margin=dict(t=60, l=10, r=10, b=10))
fig.show()

por_region = valor_liga.groupby('league_region')['value_eur'].sum().sort_values(ascending=False)
print('Participación de cada región en el valor total:')
print((por_region / por_region.sum() * 100).round(1).astype(str).add(' %').to_string())

Participación de cada región en el valor total:
league_region
Europa               84.7 %
América del Sur       6.7 %
América del Norte     4.1 %
Asia                  1.9 %
Medio Oriente         1.3 %
Sin Clasificar        0.8 %
Oceanía               0.3 %
África                0.1 %


In [11]:
# Fig. 2.3 — Consulta 3: overall promedio por posición (club_position) y grupo de edad
c3 = df.dropna(subset=['club_position']).pivot_table(index='club_position', columns='GrupoEdad', values='overall', aggfunc='mean')[ORDEN_EDAD]
c3 = c3.sort_values('Veteran', ascending=False)
fig = mapa_calor(c3, '{:.1f}', 'YlGnBu', dict(x='', y='Posición en el club', color='Overall'))
fig.update_layout(title='Fig. 2.3 · Overall promedio por posición y grupo de edad', height=max(450, 22 * len(c3) + 120))
fig.update_xaxes(side='top')
fig.show()
orden_ok = ((c3['Veteran'] >= c3['Prime']) & (c3['Prime'] >= c3['Young'])).mean() * 100
print(f'Posiciones donde se cumple Veteran >= Prime >= Young: {orden_ok:.0f}%')

Posiciones donde se cumple Veteran >= Prime >= Young: 100%


In [12]:
# Tablas 2.1 y 2.2 — Consulta 4: margen (valor - salario) por club y por liga
margen = base.assign(margen=base['value_eur'] - base['wage_eur'])
moda = lambda s: s.mode().iat[0] if not s.mode().empty else '-'
t_club = (margen.groupby('club_name')
          .agg(Liga=('league_name', moda), Registros=('player_id', 'size'),
               Valor_total=('value_eur', 'sum'), Salario_total=('wage_eur', 'sum'), Margen=('margen', 'sum'))
          .nlargest(10, 'Margen').reset_index().rename(columns={'club_name': 'Club'}))
t_club.insert(0, 'Puesto', range(1, len(t_club) + 1))
t_liga = (margen.groupby('league_name')
          .agg(Región=('league_region', moda), Clubes=('club_name', 'nunique'),
               Margen=('margen', 'sum'), Margen_por_registro=('margen', 'mean'))
          .nlargest(10, 'Margen').reset_index().rename(columns={'league_name': 'Liga'}))
t_liga.insert(0, 'Puesto', range(1, len(t_liga) + 1))
eur_m = lambda v: f'€{v / 1e6:,.0f}M'
mostrar_tabla(t_club, {'Registros': '{:,}', 'Valor_total': eur_m, 'Salario_total': eur_m, 'Margen': eur_m},
              'Tabla 2.1 · Top 10 clubes por margen acumulado (valor − salario), 2015-2024')
mostrar_tabla(t_liga, {'Margen': eur_m, 'Margen_por_registro': lambda v: f'€{v / 1e6:,.2f}M'},
              'Tabla 2.2 · Top 10 ligas por margen acumulado (valor − salario), 2015-2024')

Puesto,Club,Liga,Registros,Valor_total,Salario_total,Margen
1,Real Madrid CF,La Liga,316,"€8,299M",€43M,"€8,257M"
2,Manchester City,Premier League,311,"€7,754M",€33M,"€7,721M"
3,FC Barcelona,La Liga,299,"€7,744M",€40M,"€7,704M"
4,FC Bayern München,Bundesliga,262,"€7,372M",€25M,"€7,348M"
5,Paris Saint-Germain,Ligue 1,289,"€6,996M",€23M,"€6,973M"
6,Liverpool,Premier League,313,"€6,364M",€25M,"€6,339M"
7,Chelsea,Premier League,321,"€6,248M",€28M,"€6,220M"
8,Manchester United,Premier League,319,"€6,192M",€29M,"€6,163M"
9,Atlético de Madrid,La Liga,304,"€5,803M",€17M,"€5,787M"
10,Juventus,Serie A (Italia),283,"€5,668M",€26M,"€5,642M"


Puesto,Liga,Región,Clubes,Margen,Margen_por_registro
1,Premier League,Europa,42,"€62,747M",€9.63M
2,La Liga,Europa,46,"€53,315M",€8.79M
3,Serie A (Italia),Europa,42,"€39,705M",€7.00M
4,Bundesliga,Europa,51,"€39,115M",€6.67M
5,Ligue 1,Europa,51,"€28,768M",€5.03M
6,Liga Portugal,Europa,49,"€16,329M",€3.34M
7,Super Lig,Europa,55,"€11,713M",€2.36M
8,Liga Profesional,América del Sur,69,"€11,461M",€1.58M
9,Championship,Europa,47,"€11,043M",€1.58M
10,Eredivisie,Europa,37,"€10,551M",€2.23M


In [13]:
# Fig. 2.4 — Consulta 5: drill-down Región → Liga → Club (sunburst)
c5 = (base.assign(league_region=base['league_region'].fillna('Sin Clasificar'), league_name=base['league_name'].fillna('Sin Liga'))
      .groupby(['league_region', 'league_name', 'club_name'])
      .agg(Registros=('player_id', 'size'), Valor_prom=('value_eur', 'mean'), Salario_prom=('wage_eur', 'mean'))
      .reset_index())
c5 = c5[c5['Registros'] >= 20]   # clubes con al menos 20 registros, para que el gráfico se pueda leer
c5['Valor promedio (M€)'] = c5['Valor_prom'] / 1e6
fig = px.sunburst(c5, path=['league_region', 'league_name', 'club_name'], values='Registros',
                  color='Valor promedio (M€)', color_continuous_scale='YlGnBu', maxdepth=2)
fig.update_traces(hovertemplate='<b>%{label}</b><br>Registros: %{value:,}<br>Valor promedio: €%{color:.2f}M<extra></extra>')
fig.update_layout(title='Fig. 2.4 · Drill-down Región → Liga → Club (clic para bajar de nivel; el color es el valor promedio)',
                  height=650, margin=dict(t=60, l=10, r=10, b=10))
fig.show()

drill = (base.groupby('league_region').agg(Valor_prom=('value_eur', 'mean'), Salario_prom=('wage_eur', 'mean'))
         .sort_values('Valor_prom', ascending=False).reset_index().rename(columns={'league_region': 'Región'}))
mostrar_tabla(drill, {'Valor_prom': fmt_eur, 'Salario_prom': lambda v: f'€{v:,.0f}'},
              'Nivel superior del drill-down: valor y salario semanal promedio por región')

# Tabla 2.3 — Medición de tiempos (Etapa 2, Paso 6, MySQL)
tiempos = pd.DataFrame([('Sin índice adicional', 7.57), ('Con índice idx_club_id en la FK', 7.05)], columns=['Escenario', 'Tiempo promedio (ms)'])
tiempos['Mejora'] = ['-', f'{(1 - 7.05 / 7.57) * 100:.1f}%']
mostrar_tabla(tiempos, {'Tiempo promedio (ms)': '{:.2f}'}, 'Tabla 2.3 · Consulta 2 en MySQL: tiempo promedio de 3 corridas')

Región,Valor_prom,Salario_prom
Europa,€2.8M,"€12,512"
Sin Clasificar,€2.4M,"€4,457"
América del Norte,€1.6M,"€7,902"
América del Sur,€1.4M,"€6,073"
África,€1.2M,"€6,697"
Medio Oriente,€1.2M,"€8,856"
Asia,€815K,"€3,452"
Oceanía,€570K,"€2,897"


Escenario,Tiempo promedio (ms),Mejora
Sin índice adicional,7.57,-
Con índice idx_club_id en la FK,7.05,6.9%


In [14]:
# Fig. 2.5 — Pipeline de carga temporada por temporada (Etapa 2, Paso 4.2)
# Se repite la lógica del pipeline: cada temporada se agrega como un lote y se comprueba que el lote
# no repita jugadores (granularidad jugador-temporada) y que el acumulado no tenga duplicados.
pipeline = (df.groupby('Temporada')
              .agg(Registros=('player_id', 'size'), Jugadores_unicos=('player_id', 'nunique'))
              .reset_index())
pipeline['Duplicados en el lote'] = pipeline['Registros'] - pipeline['Jugadores_unicos']
pipeline['Acumulado'] = pipeline['Registros'].cumsum()
assert (pipeline['Duplicados en el lote'] == 0).all(), 'Hay un lote con jugadores repetidos'
assert pipeline['Acumulado'].iat[-1] == len(df) and not df.duplicated(subset=['player_id', 'Temporada']).any()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=pipeline['Temporada'], y=pipeline['Registros'], name='Registros cargados en la temporada', marker_color='#2563eb',
            text=pipeline['Registros'].map('{:,}'.format), textposition='outside')
fig.add_scatter(x=pipeline['Temporada'], y=pipeline['Acumulado'], name='Acumulado en la tabla', mode='lines+markers',
                line=dict(color='#0f172a', width=2), secondary_y=True)
fig.update_layout(title='Fig. 2.5 · Pipeline de carga: registros por temporada y acumulado (0 duplicados en cada lote)',
                  height=420, legend=dict(orientation='h', y=-0.2))
fig.update_yaxes(title_text='Registros de la temporada', range=[0, pipeline['Registros'].max() * 1.2], secondary_y=False)
fig.update_yaxes(title_text='Registros acumulados', range=[0, len(df) * 1.9], showgrid=False, secondary_y=True)   # la línea queda debajo de las etiquetas
fig.update_xaxes(title_text='Temporada', dtick=1)
fig.show()
print(f"Total cargado: {pipeline['Acumulado'].iat[-1]:,} registros (el mismo total que verificó MySQL en la Etapa 2) | "
      f"duplicados (player_id, Temporada): {int(pipeline['Duplicados en el lote'].sum())}")


Total cargado: 175,872 registros (el mismo total que verificó MySQL en la Etapa 2) | duplicados (player_id, Temporada): 0


**Interpretación de los resultados.**

- **Fig. 2.1:** cada fila es un club que estuvo al menos una vez en el top 10 de valor promedio, y cada celda su puesto en esa temporada (más oscuro = mejor puesto). Las filas completas de arriba son los clubes que nunca salieron del top 10 en la década, que en la Etapa 2 fueron Barcelona, Bayern, Real Madrid, PSG y Manchester City. Las filas con huecos muestran clubes que entran y salen, algo que en una tabla de 100 filas no se ve.
- **Fig. 2.2:** el área de cada rectángulo es proporcional al valor total, así que el dominio de Europa (el 84.7 % del total según la Etapa 2) se ve de inmediato, y dentro de Europa se distinguen las cinco grandes ligas.
- **Fig. 2.3:** el color sube de izquierda a derecha en casi todas las filas: los veteranos tienen el `overall` más alto en todas las posiciones, que es la conclusión de la Consulta 3.
- **Tablas 2.1 y 2.2:** el margen se deja en tabla porque lo que interesa son los montos de pocos clubes y ligas. Real Madrid, Manchester City y Barcelona encabezan, como en la Etapa 2.
- **Fig. 2.4:** el sunburst reproduce el drill-down pedido: al hacer clic en una región se ven sus ligas, y al hacer clic en una liga, sus clubes. El color indica el valor promedio en cada nivel.
- **Fig. 2.5:** cada barra es un lote del pipeline (una temporada) y la línea es el total acumulado en la tabla. Los diez lotes tienen entre 15,623 y 19,238 registros, ninguno repite un jugador dentro de su temporada, y el acumulado termina en 175,872, el mismo total que verificó MySQL en la Etapa 2. Esto confirma que la carga incremental respeta la granularidad de una fila por jugador y temporada.
- **Tabla 2.3:** el índice en la clave foránea redujo el tiempo de la consulta un 6.9 %.

En conclusión, el pipeline y las cinco consultas de la Etapa 2 quedan representadas de forma que la respuesta se lee sin recorrer tablas largas, y el sunburst permite hacer el drill-down de forma interactiva.

### **3. Etapa 3: análisis descriptivo, de diagnóstico y exploratorio.**

**Tabla 3.1** resume las medidas de tendencia central y dispersión. **Fig. 3.1** es un explorador interactivo que reemplaza los más de 20 histogramas y boxplots de la Etapa 3: se elige la variable, el grupo y la escala. Después se muestran la matriz de correlación (**Fig. 3.2**), la relación valor-overall (**Fig. 3.3**), el valor por posición y edad (**Fig. 3.4**), las tablas de anomalías (**Tablas 3.2 a 3.4**) y un ranking interactivo con los top 10 del análisis global (**Fig. 3.5**). Al final, las **Figs. 3.6 a 3.10** y las **Tablas 3.5 a 3.11** responden las preguntas obligatorias restantes: variables categóricas, pie dominante, ligas, anomalías, análisis global, reclutamiento, inversiones y profiling.

In [15]:
# Tabla 3.1 — Medidas de tendencia central y dispersión
VARS_NUM = {'age': 'Edad', 'overall': 'Overall', 'potential': 'Potencial', 'value_eur': 'Valor de mercado (€)',
            'wage_eur': 'Salario semanal (€)', 'BrechaPotencial': 'Brecha de potencial', 'pace': 'Velocidad (pace)'}
filas = []
for col, nombre in VARS_NUM.items():
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    filas.append({'Variable': nombre, 'Media': s.mean(), 'Mediana': s.median(), 'Moda': s.mode().iat[0],
                  'Mín': s.min(), 'Q1': q1, 'Q3': q3, 'Máx': s.max(), 'IQR': q3 - q1,
                  'Desv. estándar': s.std(), 'Varianza': s.var(), 'Asimetría': s.skew()})
estadisticos = pd.DataFrame(filas)
fmt_num = lambda v: f'{v:,.1f}' if abs(v) < 1e6 else f'{v:,.3g}'
mostrar_tabla(estadisticos, {c: fmt_num for c in estadisticos.columns if c != 'Variable'} | {'Asimetría': '{:.2f}'},
              'Tabla 3.1 · Medidas de tendencia central y dispersión (todas las temporadas)')

Variable,Media,Mediana,Moda,Mín,Q1,Q3,Máx,IQR,Desv. estándar,Varianza,Asimetría
Edad,25.2,25.0,22.0,16.0,21.0,28.0,47.0,7.0,4.7,21.8,0.41
Overall,65.8,66.0,64.0,40.0,61.0,70.0,94.0,9.0,7.0,49.2,0.05
Potencial,70.9,71.0,70.0,40.0,67.0,75.0,95.0,8.0,6.3,39.1,0.15
Valor de mercado (€),2.34e+06,"725,000.0",1.1e+06,"1,000.0","350,000.0",1.8e+06,1.94e+08,1.45e+06,5.95e+06,3.54e+13,8.40
Salario semanal (€),"10,639.5","4,000.0","2,000.0",500.0,"2,000.0","10,000.0","575,000.0","8,000.0","21,569.2",4.65e+08,6.62
Brecha de potencial,5.1,4.0,0.0,0.0,0.0,8.0,28.0,8.0,5.2,26.9,0.89
Velocidad (pace),68.1,69.0,69.0,21.0,63.0,75.0,97.0,12.0,10.5,110.0,-0.60


In [16]:
# Fig. 3.1 — Explorador de distribuciones (histograma + boxplot)
# Para que el notebook no pese varios MB, los histogramas y boxplots se calculan con numpy
# y a Plotly solo se le envían las barras y los cuartiles, no los 175 mil puntos.
GRUPOS = {'Ninguno': None, 'Grupo de edad': ('GrupoEdad', ORDEN_EDAD, COLOR_EDAD),
          'Posición': ('TipoPosicion', ORDEN_POS, COLOR_POS),
          'Pie dominante': ('preferred_foot', ['Right', 'Left'], {'Right': '#2563eb', 'Left': '#f59e0b'})}

def caja(s):
    q1, med, q3 = np.percentile(s, [25, 50, 75])
    iqr = q3 - q1
    return dict(q1=[q1], median=[med], q3=[q3], mean=[s.mean()],
                lowerfence=[max(s.min(), q1 - 1.5 * iqr)], upperfence=[min(s.max(), q3 + 1.5 * iqr)])

def vista_distribucion(variable, grupo, escala_log):
    datos = df[[variable] + ([GRUPOS[grupo][0]] if GRUPOS[grupo] else [])].dropna()
    x = datos[variable].astype(float)
    if escala_log:
        x = np.log10(x.clip(lower=1))
    etiqueta = ('log10 de ' if escala_log else '') + VARS_NUM[variable]
    bordes = np.histogram_bin_edges(x, bins=40)
    centros, ancho = (bordes[:-1] + bordes[1:]) / 2, np.diff(bordes)
    if GRUPOS[grupo]:
        col, orden, colores = GRUPOS[grupo]
        series = [(g, x[datos[col] == g], colores[g]) for g in orden if (datos[col] == g).any()]
    else:
        series = [('Todos', x, '#2563eb')]

    fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], subplot_titles=('Histograma', 'Boxplot (bigotes = 1.5 × IQR)'))
    for nombre, s, color in series:
        conteo, _ = np.histogram(s, bins=bordes)
        fig.add_bar(x=centros, y=conteo, width=ancho, name=nombre, marker_color=color, opacity=0.55 if len(series) > 1 else 0.85,
                    legendgroup=nombre, row=1, col=1, hovertemplate=f'{nombre}<br>%{{x:.2f}}: %{{y:,}} registros<extra></extra>')
        fig.add_trace(go.Box(name=nombre, x=[nombre], marker_color=color, legendgroup=nombre, showlegend=False, boxmean=True, **caja(s)), row=1, col=2)
    if len(series) == 1:
        s = series[0][1]
        for valor, texto, color in [(s.mean(), 'Media', '#dc2626'), (s.median(), 'Mediana', '#16a34a')]:
            fig.add_vline(x=valor, line_dash='dash', line_color=color, row=1, col=1,
                          annotation_text=f'{texto}: {valor:,.1f}', annotation_position='top right' if texto == 'Media' else 'top left')
    fig.update_layout(barmode='overlay', height=430, title=f'Fig. 3.1 · Distribución de {etiqueta}' + (f' por {grupo.lower()}' if GRUPOS[grupo] else ''),
                      legend=dict(orientation='h', y=-0.15))
    fig.update_xaxes(title_text=etiqueta, row=1, col=1)
    fig.update_yaxes(title_text='Registros', row=1, col=1)
    fig.update_yaxes(title_text=etiqueta, row=1, col=2)
    fig.show()

    resumen = pd.DataFrame([{'Grupo': n, 'Registros': len(s), 'Media': s.mean(), 'Mediana': s.median(),
                             'Desv. estándar': s.std(), 'Asimetría': s.skew()} for n, s, _ in series])
    mostrar_tabla(resumen, {'Registros': '{:,}', 'Media': '{:,.2f}', 'Mediana': '{:,.2f}', 'Desv. estándar': '{:,.2f}', 'Asimetría': '{:.2f}'})

ESTILO = {'description_width': '150px'}
w_var = widgets.Dropdown(options=[(v, k) for k, v in VARS_NUM.items()], value='age', description='Variable', style=ESTILO)
w_grupo = widgets.Dropdown(options=list(GRUPOS), value='Ninguno', description='Separar por', style=ESTILO)
w_log = widgets.Checkbox(value=False, description='Escala logarítmica (valor y salario)', indent=False)
salida = widgets.interactive_output(vista_distribucion, {'variable': w_var, 'grupo': w_grupo, 'escala_log': w_log})
display(widgets.VBox([widgets.HTML('<b>Explorador de distribuciones</b> — elija una variable y, si quiere, un grupo para compararla.'),
                      widgets.HBox([w_var, w_grupo, w_log]), salida]))

**Interpretación de los resultados.**

La Tabla 3.1 deja los estadísticos en formato de tabla porque son muchas cifras exactas por variable. La asimetría separa dos tipos de variables: edad, `overall`, potencial y velocidad son casi simétricas (asimetría cercana a 0), mientras que el valor de mercado y el salario tienen una asimetría muy positiva, porque unas pocas estrellas cobran y valen mucho más que el resto.

El explorador (Fig. 3.1) responde con una sola herramienta las preguntas de la Etapa 3 sobre distribuciones. Con la variable *Edad* se ve la moda en 22 años y la cola hacia la derecha; con *Edad* separada por *Grupo de edad*, el peso de cada grupo; con *Overall* por *Grupo de edad*, que la mediana sube de Young a Veteran; y con *Salario semanal* en escala logarítmica por *Grupo de edad* o *Posición*, que el salario sube con la edad pero que las medianas por posición son parecidas. La escala logarítmica es necesaria para el valor y el salario: sin ella, el 90 % de los registros queda comprimido en la primera barra.

En conclusión, el explorador reemplaza varias decenas de gráficos estáticos y permite que quien presenta responda en vivo cualquier pregunta sobre la forma de una variable.

In [17]:
# Fig. 3.2 — Matriz de correlación
cols_corr = ['age', 'overall', 'potential', 'value_eur', 'wage_eur', 'BrechaPotencial',
             'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
corr = df[cols_corr].corr().round(2)
fig = mapa_calor(corr, '{:.2f}', 'RdBu_r', zmin=-1, zmax=1)
fig.update_layout(title='Fig. 3.2 · Matriz de correlación de Pearson', height=560)
fig.show()
print(f"Correlación del potencial con el valor: {corr.loc['potential', 'value_eur']:.2f} | con el salario: {corr.loc['potential', 'wage_eur']:.2f}")

# Fig. 3.3 — Valor de mercado vs overall por grupo de edad (muestra de 6,000 registros)
muestra = df_val.sample(min(6000, len(df_val)), random_state=SEMILLA)
fig = px.scatter(muestra, x='overall', y='value_eur', color='GrupoEdad', color_discrete_map=COLOR_EDAD,
                 category_orders={'GrupoEdad': ORDEN_EDAD}, log_y=True, opacity=0.55,
                 hover_name='short_name', hover_data={'Temporada': True, 'club_name': True, 'age': True},
                 labels={'overall': 'Overall', 'value_eur': 'Valor de mercado (€, escala log)', 'GrupoEdad': 'Grupo de edad'})
fig.update_traces(marker=dict(size=5))
fig.update_layout(title='Fig. 3.3 · Valor de mercado vs overall por grupo de edad (muestra de 6,000 registros)', height=480)
fig.show()

# Fig. 3.4 — Valor promedio por posición y grupo de edad
c34 = df.pivot_table(index='TipoPosicion', columns='GrupoEdad', values='value_eur', aggfunc='mean').reindex(index=ORDEN_POS, columns=ORDEN_EDAD) / 1e6
fig = mapa_calor(c34, '{:.2f}', 'YlOrRd', dict(x='Grupo de edad', y='', color='M€'))
fig.update_layout(title='Fig. 3.4 · Valor de mercado promedio (M€) por posición y grupo de edad', height=380)
fig.show()

Correlación del potencial con el valor: 0.53 | con el salario: 0.50


**Interpretación de los resultados.**

- **Fig. 3.2:** el `overall` es la variable más correlacionada con el valor y el salario, y la brecha de potencial tiene una correlación negativa con la edad, porque los jóvenes son los que tienen margen de crecimiento. El potencial se relaciona un poco más con el valor que con el salario (0.53 frente a 0.50 en la Etapa 3), lo que responde a esa pregunta obligatoria.
- **Fig. 3.3:** con el eje vertical en escala logarítmica, la nube forma casi una recta: el valor crece de forma exponencial con el `overall`. Para un mismo `overall`, los jóvenes (celeste) quedan arriba y los veteranos (gris) abajo, es decir, el mercado paga más por un jugador joven del mismo nivel. Al pasar el cursor se ve el nombre y el club de cada punto.
- **Fig. 3.4:** las celdas más oscuras son los delanteros Prime y Veteran, la combinación de edad y posición con mayor valor, y la más clara es la del portero joven, igual que en la Etapa 3.

En conclusión, estos tres gráficos resumen las relaciones principales del dataset: el nivel actual determina el precio, la edad lo modula y la posición lo multiplica.

In [18]:
# Tablas 3.2 a 3.4 — Anomalías (un registro por jugador)
cols_j = ['short_name', 'Temporada', 'club_name', 'age', 'overall', 'potential']

top_brecha = (df.dropna(subset=['BrechaPotencial']).sort_values('BrechaPotencial', ascending=False)
              .drop_duplicates('player_id').head(5)[cols_j + ['BrechaPotencial']])
mostrar_tabla(top_brecha, {'BrechaPotencial': '{:.0f}', 'overall': '{:.0f}', 'potential': '{:.0f}', 'age': '{:.0f}'},
              'Tabla 3.2 · Top 5 de BrechaPotencial (potential − overall)')

subval = (df[df['wage_eur'] > 0].dropna(subset=['overall', 'wage_eur'])
          .assign(Subvaloracion=lambda d: d['overall'] / d['wage_eur'])
          .sort_values('Subvaloracion', ascending=False).drop_duplicates('player_id').head(5)[cols_j + ['wage_eur', 'Subvaloracion']])
mostrar_tabla(subval, {'wage_eur': lambda v: f'€{v:,.0f}', 'Subvaloracion': '{:.4f}', 'overall': '{:.0f}', 'potential': '{:.0f}', 'age': '{:.0f}'},
              'Tabla 3.3 · Top 5 de subvalorados (overall / salario semanal)')

p80_sal, p25_ovr = df['wage_eur'].quantile(0.80), df['overall'].quantile(0.25)
caros = (df[(df['wage_eur'] >= p80_sal) & (df['overall'] <= p25_ovr)]
         .sort_values('wage_eur', ascending=False).drop_duplicates('player_id').head(10)[cols_j + ['wage_eur']])
mostrar_tabla(caros, {'wage_eur': lambda v: f'€{v:,.0f}', 'overall': '{:.0f}', 'potential': '{:.0f}', 'age': '{:.0f}'},
              f'Tabla 3.4 · Salario alto (≥ p80 = €{p80_sal:,.0f}) y overall bajo (≤ p25 = {p25_ovr:.0f}): {len(caros)} jugadores')

short_name,Temporada,club_name,age,overall,potential,BrechaPotencial
Callum Burton,2015,Shrewsbury Town,17,50,78,28
Joel Latibeaudiere,2018,Manchester City,17,47,73,26
A. Aktaş,2017,FC Sochaux-Montbéliard,16,52,78,26
Marcus Edwards,2017,Tottenham Hotspur,17,58,84,26
Angel Gomes,2018,Manchester United,16,63,89,26


short_name,Temporada,club_name,age,overall,potential,wage_eur,Subvaloracion
Anatoliy Trubin,2024,SL Benfica,22,77,79,€500,0.1540
A. Pyatov,2021,Shakhtar Donetsk,36,76,76,€500,0.1520
Adam Hložek,2022,AC Sparta Praha,18,76,87,€500,0.1520
Moisés Ramírez,2024,Independiente DV,23,75,77,€500,0.1500
Artem Bondarenko,2024,Shakhtar Donetsk,23,75,75,€500,0.1500


short_name,Temporada,club_name,age,overall,potential,wage_eur
Pape Souaré,2024,Motherwell,33,61,74,"€33,143"
Callum McManaman,2024,Wigan Athletic,32,61,73,"€20,714"
Ji Dong Won,2024,FC Seoul,32,57,73,"€19,375"
Elias Kachunga,2024,Cambridge United,31,61,71,"€15,444"
Harry Wilson,2018,Liverpool,20,61,79,"€15,000"
Joe Garner,2024,Carlisle United,35,61,68,"€14,000"
O. Bouy,2021,Leeds United,27,61,62,"€13,000"


In [19]:
# Fig. 3.5 — Ranking interactivo con los top 10 del análisis global de la Etapa 3
def _jugadores(datos, col):
    return datos.sort_values(col, ascending=False).drop_duplicates('player_id').head(10)

q75_ovr, q25_val = df['overall'].quantile(0.75), df['value_eur'].quantile(0.25)
RANKINGS = {
    'Jugadores por valor de mercado': lambda: (_jugadores(df.dropna(subset=['value_eur']), 'value_eur'), 'short_name', 'value_eur', 'Valor máximo (€)'),
    'Jugadores por potencial': lambda: (_jugadores(df, 'potential'), 'short_name', 'potential', 'Potencial máximo'),
    'Jóvenes (Young) con overall > 80': lambda: (_jugadores(df[(df['GrupoEdad'] == 'Young') & (df['overall'] > 80)], 'overall'), 'short_name', 'overall', 'Overall'),
    'Gangas: overall alto y valor bajo': lambda: (df[(df['overall'] >= q75_ovr) & (df['value_eur'] <= q25_val)]
                                                 .sort_values(['overall', 'value_eur'], ascending=[False, True]).drop_duplicates('player_id').head(10),
                                                 'short_name', 'overall', 'Overall'),
    'Clubes con más registros': lambda: (df[df['club_name'] != 'Sin Club'].groupby('club_name').size().rename('Registros')
                                         .nlargest(10).reset_index(), 'club_name', 'Registros', 'Registros'),
    'Clubes por overall promedio (≥ 50 registros)': lambda: (df[df['club_name'] != 'Sin Club'].groupby('club_name')['overall'].agg(['mean', 'size'])
                                                            .query('size >= 50').nlargest(10, 'mean').reset_index()
                                                            .rename(columns={'mean': 'Overall promedio', 'size': 'Registros'}),
                                                            'club_name', 'Overall promedio', 'Overall promedio'),
    'Nacionalidades por overall promedio (≥ 100 registros)': lambda: (df.groupby('nationality')['overall'].agg(['mean', 'size'])
                                                                     .query('size >= 100').nlargest(10, 'mean').reset_index()
                                                                     .rename(columns={'mean': 'Overall promedio', 'size': 'Registros'}),
                                                                     'nationality', 'Overall promedio', 'Overall promedio'),
    'Nacionalidades por valor de mercado total': lambda: (df.groupby('nationality')['value_eur'].sum().rename('Valor total (€)')
                                                         .nlargest(10).reset_index(), 'nationality', 'Valor total (€)', 'Valor total (€)'),
}

def vista_ranking(nombre):
    t, etiqueta, metrica, titulo_metrica = RANKINGS[nombre]()
    t = t.reset_index(drop=True)
    es_euro = '€' in titulo_metrica
    texto = t[metrica].map(fmt_eur) if es_euro else t[metrica].map(lambda v: f'{v:,.1f}' if v % 1 else f'{v:,.0f}')
    color = t['GrupoEdad'].map(COLOR_EDAD).fillna('#2563eb') if 'GrupoEdad' in t else '#2563eb'
    fig = go.Figure(go.Bar(x=t[metrica][::-1], y=t[etiqueta][::-1], orientation='h', text=texto[::-1], textposition='outside',
                           marker_color=color[::-1] if not isinstance(color, str) else color))
    rango = [t[metrica].min() * 0.95, t[metrica].max() * 1.04] if not es_euro and metrica != 'Registros' else [0, t[metrica].max() * 1.15]
    fig.update_layout(title=f'Fig. 3.5 · Top 10: {nombre.lower()}', height=440, xaxis=dict(title=titulo_metrica, range=rango),
                      margin=dict(l=10, r=30, t=60, b=40))
    fig.show()
    if 'GrupoEdad' in t:
        print('Color de la barra = grupo de edad (celeste Young, naranja Prime, gris Veteran).')
    cols = [c for c in [etiqueta, 'Temporada', 'club_name', 'nationality', 'age', 'overall', 'potential', 'value_eur', 'Registros']
            if c in t.columns and c != metrica] + [metrica]
    formatos = {c: fmt_eur for c in cols if c in ('value_eur', 'Valor total (€)')}
    formatos |= {c: '{:.0f}' for c in ('age', 'overall', 'potential') if c in cols and c != metrica}
    formatos |= {'Overall promedio': '{:.2f}'} if 'Overall promedio' in cols else {}
    mostrar_tabla(t[cols].rename(columns={'short_name': 'Jugador', 'club_name': 'Club', 'nationality': 'Nacionalidad'}),
                  {({'short_name': 'Jugador', 'club_name': 'Club', 'nationality': 'Nacionalidad'}).get(k, k): v for k, v in formatos.items()})

w_rank = widgets.Dropdown(options=list(RANKINGS), value='Jugadores por valor de mercado', description='Ranking',
                          style=ESTILO, layout=widgets.Layout(width='520px'))
display(widgets.VBox([widgets.HTML('<b>Rankings del análisis global</b> — elija el ranking que quiere ver.'),
                      w_rank, widgets.interactive_output(vista_ranking, {'nombre': w_rank})]))

**Interpretación de los resultados.**

Las anomalías (Tablas 3.2 a 3.4) se dejan en tabla porque cada fila es un jugador que hay que poder identificar y revisar: la Tabla 3.2 muestra a los jóvenes con mayor margen de crecimiento (Callum Burton encabeza con 28 puntos), la Tabla 3.3 a los jugadores que más `overall` dan por cada euro de salario y la Tabla 3.4 a los pocos jugadores que cobran mucho para su nivel.

El ranking interactivo (Fig. 3.5) junta en una sola vista los top 10 del análisis global de la Etapa 3. Cada ranking combina tabla y barra: la tabla da el nombre y el contexto (club, temporada, edad), y la barra muestra si el primer puesto se separa del resto o si están empatados, por ejemplo, el potencial máximo, donde varios jugadores comparten el mismo valor.

En conclusión, las preguntas sobre jugadores concretos se responden con tablas y las comparaciones con barras, y el selector permite pasar de una pregunta a otra durante la presentación sin desplazarse por el notebook.

#### **Preguntas obligatorias de la Etapa 3 que no cubren el explorador ni el ranking.**

El explorador (Fig. 3.1) y el ranking (Fig. 3.5) responden la mayoría de las preguntas de la Etapa 3, pero varias preguntas obligatorias necesitan su propia representación. Para cada una se usa la misma definición y los mismos filtros de la Etapa 3 (por ejemplo, solo registros con una posición real de campo cuando la pregunta es por grupo de posición), de modo que las respuestas coinciden.

**Análisis univariante categórico y bivariante.** La **Fig. 3.6** reúne los gráficos de barras categóricos, la **Fig. 3.7** y la **Tabla 3.5** responden las preguntas sobre el pie dominante, y la **Fig. 3.8** la liga con mayor valor promedio.

In [20]:
# Fig. 3.6 — Univariante categórico (Etapa 3, Paso 2.2): top 10 nacionalidades y clubes,
# y distribución por TipoPosicion, GrupoEdad y pie dominante
def agrupar_posicion(pos):
    # Misma agrupación de club_position que la Etapa 3 (Pasos 4 a 6)
    if pos in ['CB', 'LCB', 'RCB', 'LB', 'RB', 'LWB', 'RWB']:
        return 'Defensores'
    if pos in ['CM', 'LCM', 'RCM', 'CDM', 'LDM', 'RDM', 'CAM', 'LAM', 'RAM', 'LM', 'RM']:
        return 'Mediocampistas'
    if pos in ['ST', 'CF', 'LW', 'RW', 'LF', 'RF', 'LS', 'RS']:
        return 'Ofensivos'
    if pos == 'GK':
        return 'Porteros'
    return 'Otros'

# Registros con una posición real de campo (sin SUB, RES ni posición faltante), como en la Etapa 3
titulares = df[df['club_position'].notna() & ~df['club_position'].isin(['SUB', 'RES', 'Unknown'])].copy()
titulares['grupo_posicion'] = titulares['club_position'].apply(agrupar_posicion)
titulares = titulares[titulares['grupo_posicion'] != 'Otros']
COLOR_GRUPO = {'Porteros': COLOR_POS['Portero'], 'Defensores': COLOR_POS['Defensa'],
               'Mediocampistas': COLOR_POS['Mediocampista'], 'Ofensivos': COLOR_POS['Delantero']}
COLOR_PIE = {'Right': '#2563eb', 'Left': '#f59e0b'}

top_nac = df['nationality'].value_counts().head(10)
top_club = df.loc[df['club_name'] != 'Sin Club', 'club_name'].value_counts().head(10)
dist_tipo = df['TipoPosicion'].value_counts().reindex(ORDEN_POS)
dist_edad = df['GrupoEdad'].value_counts().reindex(ORDEN_EDAD)
dist_pie = df['preferred_foot'].value_counts().reindex(['Right', 'Left'])

fig = make_subplots(rows=2, cols=6, vertical_spacing=0.2, horizontal_spacing=0.09,
                    specs=[[{'colspan': 3}, None, None, {'colspan': 3}, None, None],
                           [{'colspan': 2}, None, {'colspan': 2}, None, {'colspan': 2}, None]],
                    subplot_titles=('Top 10 nacionalidades', "Top 10 clubes (sin 'Sin Club')",
                                    'TipoPosicion', 'GrupoEdad', 'Pie dominante'))
for serie, fila, col in [(top_nac, 1, 1), (top_club, 1, 4)]:
    fig.add_bar(x=serie.values[::-1], y=serie.index[::-1], orientation='h', marker_color='#2563eb', showlegend=False,
                text=[f'{v:,}' for v in serie.values[::-1]], textposition='outside', row=fila, col=col,
                hovertemplate='%{y}: %{x:,} registros<extra></extra>')
    fig.update_xaxes(range=[0, serie.max() * 1.25], row=fila, col=col)
for serie, colores, col in [(dist_tipo, COLOR_POS, 1), (dist_edad, COLOR_EDAD, 3), (dist_pie, COLOR_PIE, 5)]:
    pct = serie / serie.sum() * 100
    fig.add_bar(x=serie.index, y=serie.values, marker_color=[colores[k] for k in serie.index], showlegend=False,
                text=[f'{p:.1f}%' for p in pct], textposition='outside', row=2, col=col,
                customdata=pct.values, hovertemplate='%{x}: %{y:,} registros (%{customdata:.1f}%)<extra></extra>')
    fig.update_yaxes(range=[0, serie.max() * 1.2], row=2, col=col)
fig.update_layout(title='Fig. 3.6 · Variables categóricas: registros jugador-temporada (2015-2024)', height=760,
                  margin=dict(t=90, l=10, r=20, b=40))
fig.show()
print(f"Nacionalidad con más registros: {top_nac.index[0]} ({top_nac.iat[0]:,}) | club con más registros: {top_club.index[0]} ({top_club.iat[0]:,})")
print('Distribución por GrupoEdad:', (dist_edad / dist_edad.sum() * 100).round(1).astype(str).add(' %').to_dict())


Nacionalidad con más registros: England (16,235) | club con más registros: Arsenal (342)
Distribución por GrupoEdad: {'Young': '18.0 %', 'Prime': '57.1 %', 'Veteran': '25.0 %'}


In [21]:
# Fig. 3.7 — Pie dominante: overall de diestros y zurdos (bivariante, Preg. 4), zurdos por grupo de posición
# (multivariante, Preg. 1) e influencia del pie dominante (profiling)
zurdos = (titulares.groupby('grupo_posicion')['preferred_foot']
          .agg(Registros='size', Zurdos=lambda s: (s == 'Left').sum())
          .reindex(['Porteros', 'Defensores', 'Mediocampistas', 'Ofensivos']))
zurdos['% zurdos'] = zurdos['Zurdos'] / zurdos['Registros'] * 100

fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.12,
                    subplot_titles=('Overall por pie dominante (bigotes = 1.5 × IQR)', '% de zurdos por grupo de posición (titulares)'))
for pie, nombre in [('Right', 'Diestros'), ('Left', 'Zurdos')]:
    fig.add_trace(go.Box(name=nombre, x=[nombre], marker_color=COLOR_PIE[pie], boxmean=True, showlegend=False,
                         **caja(df.loc[df['preferred_foot'] == pie, 'overall'].dropna())), row=1, col=1)
fig.add_bar(x=zurdos.index, y=zurdos['% zurdos'], marker_color=[COLOR_GRUPO[g] for g in zurdos.index], showlegend=False,
            text=[f'{p:.1f}%' for p in zurdos['% zurdos']], textposition='inside', insidetextanchor='end', textfont=dict(color='white'),
            customdata=np.stack([zurdos['Zurdos'], zurdos['Registros']], axis=-1),
            hovertemplate='%{x}: %{y:.1f}% zurdos<br>%{customdata[0]:,} de %{customdata[1]:,} registros<extra></extra>', row=1, col=2)
fig.add_hline(y=(df['preferred_foot'] == 'Left').mean() * 100, line_dash='dash', line_color='#64748b', row=1, col=2,
              annotation_text=f"Todo el dataset: {(df['preferred_foot'] == 'Left').mean() * 100:.1f}%", annotation_position='top right')
fig.update_yaxes(title_text='Overall', row=1, col=1)
fig.update_yaxes(title_text='% de zurdos', range=[0, zurdos['% zurdos'].max() * 1.3], row=1, col=2)
fig.update_layout(title='Fig. 3.7 · Pie dominante: rendimiento y presencia de zurdos por posición', height=430)
fig.show()

perfil_pie = (df.groupby('preferred_foot')
                .agg(Registros=('player_id', 'size'), Overall_medio=('overall', 'mean'), Overall_mediano=('overall', 'median'),
                     Valor_medio=('value_eur', 'mean'), Salario_medio=('wage_eur', 'mean'), Salario_mediano=('wage_eur', 'median'))
                .reindex(['Right', 'Left']).reset_index())
perfil_pie.insert(2, '% del total', perfil_pie['Registros'] / perfil_pie['Registros'].sum() * 100)
perfil_pie['preferred_foot'] = perfil_pie['preferred_foot'].map({'Right': 'Diestros', 'Left': 'Zurdos'})
mostrar_tabla(perfil_pie.rename(columns={'preferred_foot': 'Pie dominante'}),
              {'Registros': '{:,}', '% del total': '{:.1f}%', 'Overall_medio': '{:.2f}', 'Overall_mediano': '{:.0f}',
               'Valor_medio': fmt_eur, 'Salario_medio': lambda v: f'€{v:,.0f}', 'Salario_mediano': lambda v: f'€{v:,.0f}'},
              'Tabla 3.5 · Influencia del pie dominante (todos los registros)')


Pie dominante,Registros,% del total,Overall_medio,Overall_mediano,Valor_medio,Salario_medio,Salario_mediano
Diestros,"134,483",76.5%,65.60,66,€2.3M,"€10,500","€4,000"
Zurdos,"41,389",23.5%,66.31,66,€2.5M,"€11,094","€4,000"


In [22]:
# Fig. 3.8 — Liga con el valor de mercado promedio más alto (bivariante, Preg. 5)
# Se usan todas las ligas con al menos 1,000 registros, para que el promedio sea estable.
MIN_REG_LIGA = 1000
ligas = (df.dropna(subset=['value_eur']).groupby('league_name')
           .agg(Region=('league_region', lambda s: s.mode().iat[0] if not s.mode().empty else 'Sin Clasificar'),
                Registros=('player_id', 'size'), Valor_promedio=('value_eur', 'mean'), Valor_mediano=('value_eur', 'median'))
           .query('Registros >= @MIN_REG_LIGA').nlargest(10, 'Valor_promedio').reset_index())
fig = px.bar(ligas.iloc[::-1], x=ligas['Valor_promedio'].iloc[::-1] / 1e6, y='league_name', orientation='h', color='Region',
             text=ligas['Valor_promedio'].iloc[::-1].map(fmt_eur), color_discrete_sequence=px.colors.qualitative.Safe,
             labels={'x': 'Valor de mercado promedio (M€)', 'league_name': '', 'Region': 'Región'},
             hover_data={'Registros': ':,', 'Valor_mediano': ':,.0f'})
fig.update_traces(textposition='outside')
fig.update_yaxes(categoryorder='total ascending')   # al colorear por región, se mantiene el orden por valor
fig.update_layout(title=f'Fig. 3.8 · Top 10 ligas por valor de mercado promedio (ligas con ≥ {MIN_REG_LIGA:,} registros)', height=440,
                  xaxis=dict(range=[0, ligas['Valor_promedio'].max() / 1e6 * 1.2]))
fig.show()
print(f"Ligas evaluadas: {(df.groupby('league_name').size() >= MIN_REG_LIGA).sum()} de {df['league_name'].nunique()} | "
      f"liga con mayor valor promedio: {ligas.at[0, 'league_name']} ({fmt_eur(ligas.at[0, 'Valor_promedio'])}), "
      f"seguida de {ligas.at[1, 'league_name']} ({fmt_eur(ligas.at[1, 'Valor_promedio'])})")


Ligas evaluadas: 39 de 62 | liga con mayor valor promedio: Premier League (€9.7M), seguida de La Liga (€8.8M)


**Interpretación de los resultados.**

La Fig. 3.6 reúne los cinco gráficos de barras del análisis categórico. England es la nacionalidad con más registros (16,235), muy por encima de Germany (10,337) y Spain (10,295), y Arsenal es el club con más registros (342), aunque los diez primeros clubes están entre 318 y 342, casi empatados. Por posición, los mediocampistas (37.0 %) y los defensas (32.8 %) son la mayoría; por edad, el grupo Prime reúne el 57.1 % de los registros, y el 76.5 % de los registros son de jugadores diestros.

La Fig. 3.7 responde las preguntas sobre el pie dominante. Los boxplots de diestros y zurdos están a la misma altura (mediana de 66 en los dos), así que el pie no cambia el rendimiento. Lo que sí cambia es dónde juegan los zurdos: son el 33.1 % de los defensores titulares, por los laterales y centrales izquierdos, y solo el 12.4 % de los porteros, frente al 23.5 % de todo el dataset. La Tabla 3.5 completa el perfil: los zurdos tienen un `overall`, un valor y un salario medios algo mayores, pero las medianas de `overall` y de salario son iguales, así que la influencia del pie dominante es mínima.

La Fig. 3.8 compara las 39 ligas con al menos 1,000 registros. La Premier League tiene el valor promedio más alto (€9.7M), seguida de La Liga (€8.8M), la Serie A (€7.0M) y la Bundesliga (€6.7M), la misma respuesta de la Etapa 3. El color muestra que nueve de las diez ligas son europeas; solo la Serie A de Brasil entra desde América del Sur.

En conclusión, las preguntas categóricas se responden con barras porque comparan frecuencias o promedios entre pocas categorías, y el boxplot muestra de un vistazo que diestros y zurdos rinden igual.

**Análisis multivariante y detección de anomalías.** La **Tabla 3.6** responde quiénes son los jóvenes mejor pagados por posición, la **Tabla 3.7** resume los métodos Z-score e IQR con sus anomalías más extremas, y la **Tabla 3.8** recoge el análisis manual de tres registros. Se usan tablas porque en las tres hay que identificar a cada jugador y leer sus valores exactos.

In [23]:
# Tabla 3.6 — Jóvenes mejor pagados por grupo de posición (multivariante, Preg. 2)
jov = titulares[titulares['GrupoEdad'] == 'Young']
mejor_pagado = (jov.loc[jov.groupby('grupo_posicion')['wage_eur'].idxmax()]
                  .sort_values('wage_eur', ascending=False)
                  [['grupo_posicion', 'short_name', 'club_name', 'Temporada', 'age', 'overall', 'wage_eur']])
mostrar_tabla(mejor_pagado.rename(columns={'grupo_posicion': 'Grupo de posición', 'short_name': 'Jugador', 'club_name': 'Club',
                                           'age': 'Edad', 'overall': 'Overall', 'wage_eur': 'Salario semanal'}),
              {'Edad': '{:.0f}', 'Overall': '{:.0f}', 'Salario semanal': lambda v: f'€{v:,.0f}'},
              'Tabla 3.6 · Joven (Young) mejor pagado de cada grupo de posición')

# Tabla 3.7 — Resumen de los métodos de detección de anomalías (Z-score en 2 variables e IQR en 1)
def casos(datos, col, n=3):
    # Los n jugadores más extremos, uno por jugador
    t = datos.sort_values(col, ascending=False).drop_duplicates('player_id').head(n)
    return ', '.join(f"{r.short_name} ({getattr(r, col):,.0f})" for r in t.itertuples())

filas_anom = []
for col, nombre in [('age', 'Edad'), ('wage_eur', 'Salario semanal')]:
    s = df[col].dropna()
    z = (df[col] - s.mean()) / s.std()
    marcados = df[z.abs() > 3]
    limite = s.mean() + 3 * s.std()
    texto_lim = f'> €{limite:,.0f}' if col == 'wage_eur' else f'> {limite:.1f} años'
    filas_anom.append(('Z-score (|z| > 3)', nombre, texto_lim, len(marcados), len(marcados) / len(df) * 100,
                       casos(marcados, col)))
q1_b, q3_b = df['BrechaPotencial'].quantile([0.25, 0.75])
lim_sup_b = q3_b + 1.5 * (q3_b - q1_b)
marcados_b = df[(df['BrechaPotencial'] > lim_sup_b) | (df['BrechaPotencial'] < q1_b - 1.5 * (q3_b - q1_b))]
filas_anom.append(('IQR (1.5 × IQR)', 'BrechaPotencial', f'> {lim_sup_b:.0f} puntos', len(marcados_b), len(marcados_b) / len(df) * 100,
                   casos(marcados_b, 'BrechaPotencial')))
anom = pd.DataFrame(filas_anom, columns=['Método', 'Variable', 'Límite', 'Registros anómalos', '% del total', 'Casos más extremos'])
mostrar_tabla(anom, {'Registros anómalos': '{:,}', '% del total': '{:.2f}%'},
              'Tabla 3.7 · Detección de anomalías: Z-score en edad y salario, IQR en BrechaPotencial')

# Tabla 3.8 — Análisis manual de tres registros (misma selección de la Etapa 3)
flag_inc = a_bool(df['flag_valor_salario_inconsistente'])
manual = pd.concat([
    df.sort_values('wage_eur', ascending=False).head(1).assign(Tipo='Estrella',
        Decisión='Valor extremo real: máximo salario, overall 94 y club de gran presupuesto.',
        Acción='Conservar; usar mediana o IQR en los estadísticos generales.'),
    df.sort_values('BrechaPotencial', ascending=False).head(1).assign(Tipo='Gema',
        Decisión='Mayor brecha del dataset; el potencial de una sola temporada es una estimación.',
        Acción='Conservar; seguir su evolución antes de usarlo para reclutar.'),
    df[flag_inc].sort_values('wage_eur', ascending=False).head(1).assign(Tipo='Posible error',
        Decisión='Salario semanal casi igual a su valor de mercado: valor subestimado.',
        Acción='Conservar como sospechoso y excluir de cálculos con la razón valor/salario.'),
])[['Tipo', 'short_name', 'club_name', 'Temporada', 'age', 'overall', 'potential', 'value_eur', 'wage_eur', 'BrechaPotencial', 'Decisión', 'Acción']]
mostrar_tabla(manual.rename(columns={'short_name': 'Jugador', 'club_name': 'Club', 'age': 'Edad', 'overall': 'Overall',
                                     'potential': 'Potencial', 'value_eur': 'Valor', 'wage_eur': 'Salario semanal', 'BrechaPotencial': 'Brecha'}),
              {'Edad': '{:.0f}', 'Overall': '{:.0f}', 'Potencial': '{:.0f}', 'Brecha': '{:.0f}', 'Valor': lambda v: f'€{v:,.0f}',
               'Salario semanal': lambda v: f'€{v:,.0f}'},
              'Tabla 3.8 · Análisis manual de anomalías: clasificación y acción')


Grupo de posición,Jugador,Club,Temporada,Edad,Overall,Salario semanal
Ofensivos,Ousmane Dembélé,FC Barcelona,2018,20,83,"€150,000"
Mediocampistas,Pedri,FC Barcelona,2023,19,85,"€115,000"
Defensores,Luke Shaw,Manchester United,2017,20,81,"€90,000"
Porteros,Gianluigi Donnarumma,AC Milan,2020,20,85,"€34,000"


Método,Variable,Límite,Registros anómalos,% del total,Casos más extremos
Z-score (|z| > 3),Edad,> 39.2 años,227,0.13%,"B. Richardson (47), Kim Byung Ji (45), Ó. Pérez (45)"
Z-score (|z| > 3),Salario semanal,"> €75,347","3,447",1.96%,"Cristiano Ronaldo (575,000), Lionel Messi (575,000), L. Suárez (525,000)"
IQR (1.5 × IQR),BrechaPotencial,> 20 puntos,816,0.46%,"Callum Burton (28), Marcus Edwards (26), A. Aktaş (26)"


Tipo,Jugador,Club,Temporada,Edad,Overall,Potencial,Valor,Salario semanal,Brecha,Decisión,Acción
Estrella,Cristiano Ronaldo,Real Madrid CF,2018,32,94,94,"€95,500,000","€575,000",0,"Valor extremo real: máximo salario, overall 94 y club de gran presupuesto.",Conservar; usar mediana o IQR en los estadísticos generales.
Gema,Callum Burton,Shrewsbury Town,2015,17,50,78,"€20,000","€2,000",28,Mayor brecha del dataset; el potencial de una sola temporada es una estimación.,Conservar; seguir su evolución antes de usarlo para reclutar.
Posible error,F. Lampard,Manchester City,2015,36,79,79,"€110,000","€80,000",0,Salario semanal casi igual a su valor de mercado: valor subestimado.,Conservar como sospechoso y excluir de cálculos con la razón valor/salario.


**Interpretación de los resultados.**

La Tabla 3.6 muestra al joven (Young) mejor pagado de cada grupo de posición: Ousmane Dembélé entre los ofensivos (€150,000 semanales en 2018), Pedri entre los mediocampistas (€115,000), Luke Shaw entre los defensores (€90,000) y Gianluigi Donnarumma entre los porteros (€34,000). Los jóvenes de posiciones ofensivas son los que más cobran, y los porteros ganan menos aunque sean de élite.

La Tabla 3.7 resume los tres métodos de detección. El Z-score marca 227 registros por edad, todos de 40 años o más, y 3,447 por salario, por encima de unos €75,000 semanales; el IQR marca 816 registros con una brecha de potencial mayor que 20 puntos, todos de jugadores jóvenes. Los casos más extremos (Cristiano Ronaldo y Lionel Messi en salario, Callum Burton en brecha) son jugadores reales y no errores. La Tabla 3.8 deja por escrito la decisión del análisis manual: la estrella y la gema se conservan, y el registro de F. Lampard, con un salario de €80,000 semanales frente a un valor de €110,000, se conserva marcado como sospechoso.

En conclusión, las anomalías se presentan en tablas porque cada fila es un jugador concreto con una decisión, y el resumen por método deja ver que el Z-score sobre el salario marca muchos registros legítimos, por la fuerte asimetría de esa variable.

**Análisis global y específico.** La **Fig. 3.9** reúne cuatro rankings del análisis global, la **Fig. 3.10** la distribución por posición y la comparación entre atacantes y defensores, y las **Tablas 3.9 a 3.11** las preguntas de reclutamiento, inversiones y profiling, donde importan los nombres y varias medidas a la vez.

In [24]:
# Fig. 3.9 — Análisis global: potencial por grupo de edad, clubes con mayor potencial promedio,
# nacionalidades con salario medio más alto y nacionalidades predominantes entre los jugadores de alto valor
pot_edad = df.groupby('GrupoEdad')['potential'].mean().reindex(ORDEN_EDAD)
clubes_pot = (df[df['club_name'] != 'Sin Club'].groupby('club_name')['potential'].agg(['mean', 'size'])
                .query('size >= 100').nlargest(10, 'mean'))
nac_validas = df['nationality'].value_counts().loc[lambda s: s >= 100].index
nac_sal = df[df['nationality'].isin(nac_validas)].groupby('nationality')['wage_eur'].mean().nlargest(10)
umbral_alto_valor = df['value_eur'].quantile(0.90)
nac_alto = df.loc[df['value_eur'] >= umbral_alto_valor, 'nationality'].value_counts().head(10)

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, horizontal_spacing=0.2,
                    subplot_titles=('Potencial promedio por grupo de edad', 'Top 10 clubes por potencial promedio (≥ 100 registros)',
                                    'Top 10 nacionalidades por salario medio (≥ 100 registros)',
                                    f'Nacionalidades entre el 10 % de mayor valor (≥ {fmt_eur(umbral_alto_valor)})'))
fig.add_bar(x=pot_edad.index, y=pot_edad.values, marker_color=[COLOR_EDAD[g] for g in pot_edad.index],
            text=[f'{v:.1f}' for v in pot_edad.values], textposition='outside', showlegend=False, row=1, col=1)
fig.update_yaxes(range=[pot_edad.min() - 4, pot_edad.max() + 2], row=1, col=1)
for serie, fila, col, fmt in [(clubes_pot['mean'], 1, 2, '{:.2f}'), (nac_sal, 2, 1, '€{:,.0f}'), (nac_alto, 2, 2, '{:,}')]:
    fig.add_bar(x=serie.values[::-1], y=serie.index[::-1], orientation='h', marker_color='#2563eb', showlegend=False,
                text=[fmt.format(v) for v in serie.values[::-1]], textposition='outside', row=fila, col=col)
fig.update_xaxes(range=[clubes_pot['mean'].min() - 3, clubes_pot['mean'].max() + 1.5], row=1, col=2)
fig.update_xaxes(range=[0, nac_sal.max() * 1.3], row=2, col=1)
fig.update_xaxes(range=[0, nac_alto.max() * 1.25], row=2, col=2)
fig.update_layout(title='Fig. 3.9 · Análisis global: potencial, salario y nacionalidades del alto valor', height=780,
                  margin=dict(t=90, l=10, r=20, b=40))
fig.show()
print(f"Grupo de edad con mayor potencial: {pot_edad.idxmax()} ({pot_edad.max():.1f}) | club con mayor potencial promedio: "
      f"{clubes_pot.index[0]} ({clubes_pot['mean'].iat[0]:.2f}) | nacionalidad con salario medio más alto: {nac_sal.index[0]} (€{nac_sal.iat[0]:,.0f}) | "
      f"nacionalidad con más registros de alto valor: {nac_alto.index[0]} ({nac_alto.iat[0]:,})")


Grupo de edad con mayor potencial: Young (72.3) | club con mayor potencial promedio: FC Barcelona (84.15) | nacionalidad con salario medio más alto: Gabon (€22,408) | nacionalidad con más registros de alto valor: Spain (2,146)


In [25]:
# Fig. 3.10 — Distribución por posición (titulares) y atacantes contra defensores (salario medio y mediano)
def grupo_rol(pos):
    # Agrupación de PosicionPrincipal que usó la Etapa 3 en "Attackers vs. Defenders"
    if pos in ['ST', 'CF', 'LF', 'RF', 'LW', 'RW', 'LS', 'RS']:
        return 'Atacantes'
    if pos in ['CB', 'LCB', 'RCB', 'LB', 'RB', 'LWB', 'RWB']:
        return 'Defensores'
    if pos == 'GK':
        return 'Porteros'
    return 'Mediocampistas'

dist_grupo = titulares['grupo_posicion'].value_counts().reindex(['Porteros', 'Defensores', 'Mediocampistas', 'Ofensivos'])
rol = df.assign(Rol=df['PosicionPrincipal'].apply(grupo_rol)).groupby('Rol')['wage_eur'].agg(['mean', 'median']) \
        .reindex(['Atacantes', 'Mediocampistas', 'Defensores', 'Porteros'])

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.12,
                    subplot_titles=('Registros por grupo de posición (titulares)', 'Salario semanal por rol: media y mediana'))
fig.add_bar(x=dist_grupo.index, y=dist_grupo.values, marker_color=[COLOR_GRUPO[g] for g in dist_grupo.index], showlegend=False,
            text=[f'{v:,}<br>({v / dist_grupo.sum() * 100:.0f}%)' for v in dist_grupo.values], textposition='outside', row=1, col=1)
fig.update_yaxes(range=[0, dist_grupo.max() * 1.25], row=1, col=1)
for medida, nombre, color in [('mean', 'Media', '#2563eb'), ('median', 'Mediana', '#94a3b8')]:
    fig.add_bar(x=rol.index, y=rol[medida], name=nombre, marker_color=color, text=[f'€{v:,.0f}' for v in rol[medida]],
                textposition='outside', row=1, col=2)
fig.update_yaxes(title_text='€ por semana', range=[0, rol['mean'].max() * 1.2], row=1, col=2)
fig.update_layout(title='Fig. 3.10 · Distribución por posición y salario de atacantes contra defensores', height=440,
                  barmode='group', legend=dict(orientation='h', y=-0.15, x=0.6))
fig.show()


In [26]:
# Tabla 3.9 — Reclutamiento: jóvenes menores de 21 años con mayor brecha de potencial (titulares, uno por jugador)
recl = (titulares[titulares['age'] < 21].sort_values('BrechaPotencial', ascending=False).drop_duplicates('player_id').head(10)
        [['short_name', 'Temporada', 'club_name', 'age', 'overall', 'potential', 'BrechaPotencial', 'value_eur']])
mostrar_tabla(recl.rename(columns={'short_name': 'Jugador', 'club_name': 'Club', 'age': 'Edad', 'overall': 'Overall',
                                   'potential': 'Potencial', 'BrechaPotencial': 'Brecha', 'value_eur': 'Valor'}),
              {'Edad': '{:.0f}', 'Overall': '{:.0f}', 'Potencial': '{:.0f}', 'Brecha': '{:.0f}', 'Valor': fmt_eur},
              'Tabla 3.9 · Top 10 jóvenes (< 21 años) con mayor brecha de potencial')

# Tabla 3.10 — Inversiones: clubes con mejor ratio valor/salario (≥ 50 registros, salario > 0, sin 'Sin Club')
fin = df[(df['wage_eur'] > 0) & (df['club_name'] != 'Sin Club')].assign(ratio=lambda d: d['value_eur'] / d['wage_eur'])
fin = fin[fin['club_name'].isin(fin['club_name'].value_counts().loc[lambda s: s >= 50].index)]
ratio_club = (fin.groupby('club_name')
                .agg(Liga=('league_name', lambda s: s.mode().iat[0]), Registros=('player_id', 'size'),
                     Ratio_medio=('ratio', 'mean'), Salario_mediano=('wage_eur', 'median'), Valor_mediano=('value_eur', 'median'))
                .nlargest(5, 'Ratio_medio').reset_index().rename(columns={'club_name': 'Club'}))
ratio_club.insert(0, 'Puesto', range(1, len(ratio_club) + 1))
mostrar_tabla(ratio_club, {'Registros': '{:,}', 'Ratio_medio': '{:,.0f}', 'Salario_mediano': lambda v: f'€{v:,.0f}', 'Valor_mediano': fmt_eur},
              'Tabla 3.10 · Top 5 clubes por ratio valor de mercado / salario semanal')

# Tabla 3.11 — Profiling: perfil del centrocampista de élite (overall ≥ 85) frente a los demás mediocampistas
df_rol = df.assign(Rol=df['PosicionPrincipal'].apply(grupo_rol))
medios = df_rol[df_rol['Rol'] == 'Mediocampistas']
cols_perfil = {'age': 'Edad', 'overall': 'Overall', 'potential': 'Potencial', 'value_eur': 'Valor', 'wage_eur': 'Salario semanal',
               'pace': 'Pace', 'shooting': 'Shooting', 'passing': 'Passing', 'dribbling': 'Dribbling', 'defending': 'Defending', 'physic': 'Physic'}
grupos_perfil = [('Élite (overall ≥ 85)', medios[medios['overall'] >= 85]),
                 ('Resto de mediocampistas', medios[medios['overall'] < 85]),
                 ('Todos los jugadores', df_rol)]
perfil_medio = pd.DataFrame([{'Grupo': n, 'Registros': len(g), **{v: g[k].mean() for k, v in cols_perfil.items()}}
                             for n, g in grupos_perfil])
mostrar_tabla(perfil_medio, {'Registros': '{:,}', 'Valor': fmt_eur, 'Salario semanal': lambda v: f'€{v:,.0f}'} |
              {c: '{:.1f}' for c in ['Edad', 'Overall', 'Potencial', 'Pace', 'Shooting', 'Passing', 'Dribbling', 'Defending', 'Physic']},
              'Tabla 3.11 · Perfil del centrocampista de élite (promedios)')


Jugador,Temporada,Club,Edad,Overall,Potencial,Brecha,Valor
Felipe Valencia,2024,Inter Miami CF,18,50,75,25,€150K
Peder Hoel Lervik,2024,Molde FK,18,56,81,25,€475K
D. Asonganyi,2019,Milton Keynes Dons,17,50,74,24,€70K
Osvaldo Cisneros,2024,Sporting KC,19,49,73,24,€150K
Serge Ngoma,2024,Red Bulls,18,58,82,24,€650K
Aidan Denholm,2024,Hearts,19,53,77,24,€250K
Kevin Paredes,2021,D.C. United,17,52,75,23,€250K
Joshua Feeney,2024,Aston Villa,18,57,80,23,€500K
Garang Kuol,2024,FC Volendam,19,62,85,23,€1.0M
Gavin Beavers,2024,Real Salt Lake,18,53,76,23,€230K


Puesto,Club,Liga,Registros,Ratio_medio,Salario_mediano,Valor_mediano
1,Dynamo Kyiv,Ukrainian Premier League,155,"5,976",€700,€3.5M
2,SK Slavia Praha,Fortuna Liga,155,"5,325",€850,€3.5M
3,Dinamo Zagreb,1. HNL,161,"4,683",€530,€2.4M
4,Shakhtar Donetsk,Ukrainian Premier League,261,"4,140","€1,000",€3.7M
5,AC Sparta Praha,Fortuna Liga,187,"3,794",€750,€2.4M


Grupo,Registros,Edad,Overall,Potencial,Valor,Salario semanal,Pace,Shooting,Passing,Dribbling,Defending,Physic
Élite (overall ≥ 85),294,27.8,86.5,87.8,€54.0M,"€173,703",72.5,76.3,83.4,84.2,63.4,72.0
Resto de mediocampistas,"64,735",24.8,65.9,71.2,€2.3M,"€10,199",68.8,56.8,62.4,66.1,50.2,62.6
Todos los jugadores,"175,872",25.2,65.8,70.9,€2.3M,"€10,639",68.1,52.6,57.2,61.7,51.6,65.0


**Interpretación de los resultados.**

La Fig. 3.9 responde cuatro preguntas del análisis global. Los jóvenes tienen el potencial promedio más alto (72.3, frente a 69.3 de los veteranos); FC Barcelona, Real Madrid CF y FC Bayern München encabezan el potencial promedio por club, con apenas 0.08 puntos de diferencia; Gabon tiene el salario medio más alto entre las nacionalidades con al menos 100 registros (€22,408), un resultado que hay que leer con cautela porque tiene pocos registros; y Spain es la nacionalidad con más registros en el 10 % de mayor valor (2,146), seguida de Brazil y France.

La Fig. 3.10 muestra que, entre los registros con una posición real de campo, los mediocampistas (37 %) y los defensores (36 %) son la mayoría y los porteros apenas el 10 %, como corresponde a una alineación. El panel de la derecha responde la comparación entre atacantes y defensores: el salario medio de los atacantes es mayor (€12,546 frente a €10,104), pero la mediana es de €4,000 en los dos grupos, así que la diferencia la explican las estrellas y no el jugador típico.

Las tablas responden las preguntas específicas. La Tabla 3.9 lista a los jóvenes menores de 21 años con más margen de crecimiento, con brechas de 23 a 25 puntos y valores de €70K a €1.0M: son las oportunidades de desarrollo para un club. La Tabla 3.10 muestra que Dynamo Kyiv tiene el mejor ratio valor/salario (5,976), pero también que los cinco primeros clubes tienen salarios medianos de €530 a €1,000 semanales, por lo que el ratio alto refleja salarios registrados muy bajos más que una gestión eficiente. La Tabla 3.11 describe al centrocampista de élite: 294 registros con 27.8 años, `overall` 86.5, €54.0M de valor y €173,703 semanales; frente al resto de los mediocampistas destaca sobre todo en passing (83.4 frente a 62.4) y dribbling (84.2 frente a 66.1).

En conclusión, con estas figuras y tablas todas las preguntas obligatorias de la Etapa 3 quedan representadas en la Etapa 5, con las mismas respuestas que se obtuvieron en ese notebook.

### **4. Etapa 4: modelos predictivos y cubos de BI.**

La **Tabla 4.1** resume las métricas de la Etapa 4 y la **Fig. 4.1** el tamaño de los clusters. Después se reentrenan los mismos modelos de la Etapa 4 (mismas variables, semillas y particiones) para mostrar lo que una tabla no puede: valor real contra predicho (**Fig. 4.2**), peso de cada variable (**Fig. 4.3**), matrices de confusión y curvas ROC (**Fig. 4.4**) y los tipos de jugador en el plano PCA (**Fig. 4.5**). Al final se recalculan los cubos de BI: el roll-up del valor por temporada (**Fig. 4.6**) y el slice de 2024 por región y grupo de edad (**Fig. 4.7**). Por último, la contratación se representa con las **Tablas 4.2 y 4.3**.

In [27]:
# Tabla 4.1 — Métricas de los modelos predictivos (citadas de la Etapa 4)
modelos = pd.DataFrame([
    ('Valor de mercado', 'Regresión lineal (log del valor), 9 variables deportivas', 'R² = 0.904 · RMSE = 0.4164 (log)', 'overall, age, potential, defending, shooting'),
    ('Valor de mercado', '+ temporada, posición y región', 'R² = 0.933', 'La temporada capta el cambio de escala de FIFA'),
    ('Alto potencial (> 85)', 'Regresión logística', 'Recall = 0.97 · Precisión = 0.20', 'Detecta casi todos, con muchas falsas alarmas'),
    ('Alto potencial (> 85)', 'Random Forest (recomendado)', 'F1 = 0.75 · PR-AUC = 0.82', 'Equilibra precisión y recall'),
    ('Salario semanal', 'Regresión lineal con variables deportivas', 'R² = 0.60', 'El overall domina'),
    ('Salario semanal', '+ región de la liga, posición y temporada', 'R² = 0.70', 'Medio Oriente paga +123 % frente a América del Norte'),
    ('Tipos de jugador', 'K-Means (k = 4) + PCA', '4 clusters', 'Ver Fig. 4.1'),
], columns=['Problema', 'Modelo', 'Métrica principal', 'Lectura'])
mostrar_tabla(modelos, titulo='Tabla 4.1 · Resultados de los modelos predictivos (Etapa 4)')

# Fig. 4.1 — Tamaño de los clusters de K-Means (Etapa 4)
clusters = pd.DataFrame({'Cluster': ['Jugadores de alto valor', 'Veteranos consolidados', 'Promesas en desarrollo', 'Jugadores de base'],
                         'Porcentaje': [21.4, 28.9, 27.6, 22.1]})
fig = px.bar(clusters, x='Porcentaje', y='Cluster', orientation='h', text=clusters['Porcentaje'].map(lambda v: f'{v:.1f}%'),
             color='Cluster', color_discrete_sequence=['#dc2626', '#475569', '#0ea5e9', '#a3a3a3'])
fig.update_layout(title='Fig. 4.1 · Tipos de jugador encontrados con K-Means (% de registros)', showlegend=False, height=330,
                  xaxis=dict(title='% de registros', range=[0, 35]), yaxis_title='')
fig.show()

Problema,Modelo,Métrica principal,Lectura
Valor de mercado,"Regresión lineal (log del valor), 9 variables deportivas",R² = 0.904 · RMSE = 0.4164 (log),"overall, age, potential, defending, shooting"
Valor de mercado,"+ temporada, posición y región",R² = 0.933,La temporada capta el cambio de escala de FIFA
Alto potencial (> 85),Regresión logística,Recall = 0.97 · Precisión = 0.20,"Detecta casi todos, con muchas falsas alarmas"
Alto potencial (> 85),Random Forest (recomendado),F1 = 0.75 · PR-AUC = 0.82,Equilibra precisión y recall
Salario semanal,Regresión lineal con variables deportivas,R² = 0.60,El overall domina
Salario semanal,"+ región de la liga, posición y temporada",R² = 0.70,Medio Oriente paga +123 % frente a América del Norte
Tipos de jugador,K-Means (k = 4) + PCA,4 clusters,Ver Fig. 4.1


#### **Reproducción de los modelos de la Etapa 4 para graficarlos.**

Para representar los modelos con gráficos hace falta tenerlos entrenados. La celda siguiente repite el código de la Etapa 4 con las mismas variables, las mismas semillas (`SEMILLA = 42` y `SEMILLA_PARTICION = 1234`) y las mismas particiones por jugador, y al final compara sus métricas con las reportadas en la Etapa 4. El Random Forest tarda alrededor de un minuto.

In [28]:
# Reproducción de los modelos de la Etapa 4 (mismas variables, semillas y particiones)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, mean_squared_error, r2_score, confusion_matrix, precision_score, recall_score,
                             f1_score, average_precision_score, roc_curve, roc_auc_score)

SEMILLA_PARTICION = 1234
dm = df_val.copy()
dm['log_value_eur'] = np.log1p(dm['value_eur'])
dm['log_wage_eur'] = np.log1p(dm['wage_eur'])

# 1) Valor de mercado: regresión lineal sobre log(valor) con 9 variables, partición 80/20 por jugador
feat_val = ['overall', 'potential', 'age', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMILLA_PARTICION)
idx_train, idx_test = next(gss.split(dm, groups=dm['player_id']))
train_v, test_v = dm.iloc[idx_train], dm.iloc[idx_test]
mod_v = Pipeline([('scaler', StandardScaler()), ('regresion', LinearRegression())]).fit(train_v[feat_val], train_v['log_value_eur'])
pred_v = mod_v.predict(test_v[feat_val])
rmse_v = np.sqrt(mean_squared_error(test_v['log_value_eur'], pred_v))
r2_v = r2_score(test_v['log_value_eur'], pred_v)

# 2) Alto potencial (> 85): logística y Random Forest, un tercio de los jugadores de cada clase a prueba
dm['potencial_alto'] = np.where(dm['potential'] > 85, 1, 0)
num_pot = ['age', 'overall', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
cat_pot = ['preferred_foot', 'TipoPosicion', 'league_region']
X_pot = pd.concat([dm[num_pot], pd.get_dummies(dm[cat_pot], drop_first=True, dtype=int)], axis=1)
y_pot = dm['potencial_alto']
jug_clase = dm.groupby('player_id')['potencial_alto'].max()
rs_p = np.random.RandomState(SEMILLA)
jug_test = []
for clase in [0, 1]:
    ids = np.sort(jug_clase[jug_clase == clase].index.values)
    rs_p.shuffle(ids)
    jug_test.extend(ids[: len(ids) // 3])
es_test = dm['player_id'].isin(jug_test).values
X_tr, X_te, y_tr, y_te = X_pot[~es_test], X_pot[es_test], y_pot[~es_test], y_pot[es_test]
mod_log = Pipeline([('scaler', StandardScaler()),
                    ('logistica', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEMILLA))]).fit(X_tr, y_tr)
mod_rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced_subsample',
                                random_state=SEMILLA, n_jobs=-1).fit(X_tr, y_tr)
clasif = {}
for nombre, m in [('Regresión logística', mod_log), ('Random Forest', mod_rf)]:
    pred, proba = m.predict(X_te), m.predict_proba(X_te)[:, 1]
    clasif[nombre] = dict(cm=confusion_matrix(y_te, pred), proba=proba, accuracy=accuracy_score(y_te, pred),
                          precision=precision_score(y_te, pred),
                          recall=recall_score(y_te, pred), f1=f1_score(y_te, pred),
                          roc_auc=roc_auc_score(y_te, proba), pr_auc=average_precision_score(y_te, proba))

# 3) Tipos de jugador: K-Means con k = 4 sobre edad, potencial y log de valor y salario
vars_km = ['age', 'potential', 'log_value_eur', 'log_wage_eur']
X_km = StandardScaler().fit_transform(dm[vars_km])
km = KMeans(n_clusters=4, random_state=SEMILLA, n_init=10).fit(X_km)
dm['Cluster'] = km.labels_
perfil = dm.groupby('Cluster').agg(Edad=('age', 'mean'), Potencial=('potential', 'mean'), Valor=('value_eur', 'median'))
etiquetas_km, resto = {}, list(perfil.index)   # mismas reglas de etiquetado de la Etapa 4
for col, nombre in [('Valor', 'Jugadores de alto valor'), ('Edad', 'Veteranos consolidados'), ('Potencial', 'Promesas en desarrollo')]:
    c = perfil.loc[resto, col].idxmax(); etiquetas_km[c] = nombre; resto.remove(c)
for c in resto:
    etiquetas_km[c] = 'Jugadores de base'
dm['Perfil'] = dm['Cluster'].map(etiquetas_km)
pca_km = PCA(n_components=2, random_state=SEMILLA)
coord = pca_km.fit_transform(X_km)
dm['PCA1'], dm['PCA2'] = coord[:, 0], coord[:, 1]
centros = pca_km.transform(km.cluster_centers_)
pct_km = dm['Perfil'].value_counts(normalize=True) * 100

lr, rf = clasif['Regresión logística'], clasif['Random Forest']
comparacion = pd.DataFrame([
    ('Valor de mercado: R²', 0.904, r2_v), ('Valor de mercado: RMSE (log)', 0.4164, rmse_v),
    ('Logística: recall', 0.97, lr['recall']), ('Logística: precisión', 0.20, lr['precision']),
    ('Random Forest: F1', 0.75, rf['f1']), ('Random Forest: PR-AUC', 0.82, rf['pr_auc']),
    ('K-Means: % alto valor', 21.4, pct_km.get('Jugadores de alto valor', np.nan)),
    ('K-Means: % veteranos consolidados', 28.9, pct_km.get('Veteranos consolidados', np.nan)),
    ('K-Means: % promesas en desarrollo', 27.6, pct_km.get('Promesas en desarrollo', np.nan)),
    ('K-Means: % jugadores de base', 22.1, pct_km.get('Jugadores de base', np.nan)),
], columns=['Métrica', 'Etapa 4', 'Reproducido'])
mostrar_tabla(comparacion, {'Etapa 4': '{:.4g}', 'Reproducido': '{:.4f}'}, 'Comparación con las métricas reportadas en la Etapa 4')

Métrica,Etapa 4,Reproducido
Valor de mercado: R²,0.904,0.9038
Valor de mercado: RMSE (log),0.4164,0.4164
Logística: recall,0.97,0.9711
Logística: precisión,0.2,0.2003
Random Forest: F1,0.75,0.7453
Random Forest: PR-AUC,0.82,0.8224
K-Means: % alto valor,21.4,21.4169
K-Means: % veteranos consolidados,28.9,28.9385
K-Means: % promesas en desarrollo,27.6,27.5804
K-Means: % jugadores de base,22.1,22.0641


In [29]:
# Fig. 4.2 — Valor real contra predicho (conjunto de prueba, muestra de 5,000 registros)
mp = test_v.assign(pred=pred_v).sample(min(5000, len(test_v)), random_state=SEMILLA)
lim = [min(mp['log_value_eur'].min(), mp['pred'].min()) - 0.2, max(mp['log_value_eur'].max(), mp['pred'].max()) + 0.2]
fig = px.scatter(mp, x='pred', y='log_value_eur', color='GrupoEdad', color_discrete_map=COLOR_EDAD, category_orders={'GrupoEdad': ORDEN_EDAD},
                 opacity=0.45, hover_name='short_name', hover_data={'Temporada': True, 'overall': True, 'value_eur': ':,.0f'},
                 labels={'pred': 'log(valor) predicho', 'log_value_eur': 'log(valor) real', 'GrupoEdad': 'Grupo de edad'})
fig.update_traces(marker=dict(size=5))
fig.add_scatter(x=lim, y=lim, mode='lines', line=dict(color='#0f172a', dash='dash'), name='Predicción perfecta')
fig.add_annotation(x=lim[0] + 0.3, y=lim[1] - 0.3, xanchor='left', showarrow=False,
                   text=f'R² = {r2_v:.3f} · RMSE = {rmse_v:.4f} (log)<br>el valor real suele estar dentro de un factor de {np.exp(rmse_v):.2f}')
fig.update_layout(title='Fig. 4.2 · Valor de mercado: real contra predicho (jugadores que el modelo no vio)', height=500)
fig.show()

# Fig. 4.3 — Peso de cada variable en el modelo de valor (coeficientes estandarizados)
coef = pd.Series(mod_v.named_steps['regresion'].coef_, index=feat_val)
coef = coef.reindex(coef.abs().sort_values().index)
top5 = set(coef.abs().sort_values(ascending=False).index[:5])
fig = go.Figure(go.Bar(x=coef.values, y=coef.index, orientation='h', text=coef.map(lambda v: f'{v:+.3f}'), textposition='outside',
                       marker_color=['#dc2626' if v in top5 else '#cbd5e1' for v in coef.index]))
fig.update_layout(title='Fig. 4.3 · Coeficientes estandarizados del modelo de valor (rojo: las 5 de mayor peso)', height=420,
                  xaxis_title='Cambio en log(valor) por una desviación estándar de la variable', margin=dict(l=10, r=40))
fig.show()

In [30]:
# Fig. 4.4 — Matrices de confusión y curvas ROC de los dos clasificadores de alto potencial
fig = make_subplots(rows=1, cols=3, column_widths=[0.28, 0.28, 0.44], horizontal_spacing=0.08,
                    subplot_titles=('Regresión logística', 'Random Forest', 'Curvas ROC'))
for col, nombre in [(1, 'Regresión logística'), (2, 'Random Forest')]:
    cm = clasif[nombre]['cm']
    fig.add_trace(go.Heatmap(z=cm, x=['Pred. ≤ 85', 'Pred. > 85'], y=['Real ≤ 85', 'Real > 85'], colorscale='Blues', showscale=False,
                             text=[[f'{v:,}' for v in fila] for fila in cm], texttemplate='%{text}', textfont=dict(size=14)), row=1, col=col)
    fig.update_yaxes(autorange='reversed', showticklabels=(col == 1), row=1, col=col)
for nombre, color in [('Regresión logística', '#64748b'), ('Random Forest', '#dc2626')]:
    fpr, tpr, _ = roc_curve(y_te, clasif[nombre]['proba'])
    fig.add_scatter(x=fpr, y=tpr, mode='lines', name=f"{nombre} (AUC {clasif[nombre]['roc_auc']:.3f})", line=dict(color=color, width=2.5), row=1, col=3)
fig.add_scatter(x=[0, 1], y=[0, 1], mode='lines', name='Azar', line=dict(color='#94a3b8', dash='dot'), row=1, col=3)
fig.update_xaxes(title_text='Tasa de falsos positivos', row=1, col=3)
fig.update_yaxes(title_text='Tasa de verdaderos positivos', row=1, col=3)
fig.update_layout(title='Fig. 4.4 · Alto potencial (> 85): matrices de confusión y curvas ROC', height=430, legend=dict(orientation='h', y=-0.25))
fig.show()
mostrar_tabla(pd.DataFrame([{'Modelo': n, 'Accuracy': d['accuracy'], 'Precisión': d['precision'], 'Recall': d['recall'], 'F1': d['f1'], 'ROC-AUC': d['roc_auc'], 'PR-AUC': d['pr_auc']}
                            for n, d in clasif.items()]), {c: '{:.3f}' for c in ['Accuracy', 'Precisión', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']},
              'Métricas de clasificación en el conjunto de prueba')

# Fig. 4.5 — Tipos de jugador en el plano PCA (muestra de 8,000 registros)
COLOR_PERFIL = {'Jugadores de alto valor': '#dc2626', 'Veteranos consolidados': '#475569', 'Promesas en desarrollo': '#0ea5e9', 'Jugadores de base': '#a3a3a3'}
mk = dm.sample(8000, random_state=SEMILLA)
fig = px.scatter(mk, x='PCA1', y='PCA2', color='Perfil', color_discrete_map=COLOR_PERFIL, opacity=0.5,
                 hover_name='short_name', hover_data={'age': True, 'potential': True, 'value_eur': ':,.0f', 'PCA1': False, 'PCA2': False},
                 labels={'PCA1': f'Componente 1 ({pca_km.explained_variance_ratio_[0] * 100:.0f}% de la varianza)',
                         'PCA2': f'Componente 2 ({pca_km.explained_variance_ratio_[1] * 100:.0f}%)'})
fig.update_traces(marker=dict(size=5))
fig.add_scatter(x=centros[:, 0], y=centros[:, 1], mode='markers', name='Centro del cluster',
                marker=dict(symbol='x', size=16, color='#0f172a', line=dict(width=2, color='white')))
fig.update_layout(title='Fig. 4.5 · Tipos de jugador (K-Means, k = 4) proyectados en dos componentes principales', height=520)
fig.show()

Modelo,Accuracy,Precisión,Recall,F1,ROC-AUC,PR-AUC
Regresión logística,0.945,0.200,0.971,0.332,0.993,0.785
Random Forest,0.992,0.695,0.804,0.745,0.996,0.822


**Interpretación de los resultados.**

La tabla de comparación confirma que los modelos reentrenados son los mismos de la Etapa 4: el R² del valor de mercado es 0.904 y su RMSE 0.4164; la regresión logística tiene un recall de 0.97 con una precisión de 0.20, y los cuatro clusters reúnen el 21.4 %, 28.9 %, 27.6 % y 22.1 % de los registros. La única diferencia es el F1 del Random Forest, 0.744 frente al 0.75 reportado, que se explica por la versión de scikit-learn y no cambia ninguna conclusión.

- **Fig. 4.2:** cada punto es un jugador que el modelo no vio al entrenar. Los puntos se agrupan sobre la diagonal, que marca la predicción perfecta, en todo el rango de valores: el modelo acierta tanto con jugadores baratos como con estrellas. El RMSE de 0.4164 en escala logarítmica equivale a que el valor real suele estar dentro de un factor de 1.52 del predicho. La dispersión es algo mayor en los valores más bajos, donde el precio depende de factores que el dataset no tiene.
- **Fig. 4.3:** el `overall` domina con un coeficiente de +1.20, mucho mayor que el de cualquier otra variable. Le siguen la edad (−0.28: a igual nivel, un jugador mayor vale menos), el potencial (+0.20), `defending` (−0.05) y `shooting` (+0.02). Son las mismas cinco variables que la Etapa 4 identificó como las que más afectan el valor de mercado; el gráfico muestra además que, después de las tres primeras, el peso de las demás es casi nulo.
- **Fig. 4.4:** las matrices de confusión muestran el tipo de error de cada modelo. Sobre 58,473 registros de prueba, con 830 de alto potencial, la regresión logística detecta 806 de los 830, pero a cambio marca 3,218 falsas alarmas. El Random Forest detecta 665 y solo marca 292 falsas alarmas. Las curvas ROC de los dos modelos casi tocan la esquina superior izquierda (AUC de 0.993 y 0.996), así que la diferencia está en el umbral de decisión y no en la capacidad de separar las clases: por eso el F1 y la PR-AUC, que miden el equilibrio entre aciertos y falsas alarmas, favorecen al Random Forest.
- **Accuracy:** el Random Forest clasifica bien el 99.2 % de los registros de prueba y la regresión logística el 94.5 %. Las dos cifras parecen excelentes, pero solo el 1.4 % de los registros de prueba (830 de 58,473) tiene un potencial mayor que 85: un modelo que respondiera siempre "no" tendría un accuracy de 98.6 % sin detectar a ningún jugador. Por eso la accuracy se reporta, como pide el enunciado, pero la comparación se hace con la precisión, el recall, el F1 y la PR-AUC.
- **Fig. 4.5:** las dos componentes principales explican el 90 % de la variación de las cuatro variables (61 % y 29 %), así que el plano representa bien los grupos. Los clusters forman regiones separadas: los jugadores de alto valor (26.7 años, `overall` 74.9, valor mediano €4.6M), las promesas en desarrollo (22.0 años, potencial 74.1, €950K), los veteranos consolidados (30.0 años, €600K) y los jugadores de base (21.6 años, `overall` 57.2, €200K).

En conclusión, los gráficos agregan a las métricas lo que una tabla no muestra: dónde se equivoca cada modelo, qué variable decide el valor y cómo se separan los perfiles de jugador.

In [31]:
# Fig. 4.6 — Roll-up: evolución del mercado por temporada
rollup = df_val.groupby('Temporada').agg(Valor_mediano=('value_eur', 'median'), Salario_mediano=('wage_eur', 'median'),
                                          Overall_prom=('overall', 'mean'))
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_scatter(x=rollup.index, y=rollup['Valor_mediano'] / 1e6, name='Valor mediano (M€)', mode='lines+markers', line=dict(color='#dc2626', width=3))
fig.add_scatter(x=rollup.index, y=rollup['Overall_prom'], name='Overall promedio', mode='lines+markers', line=dict(color='#2563eb', dash='dot'), secondary_y=True)
fig.update_layout(title='Fig. 4.6 · Roll-up por temporada: el valor sube y el nivel casi no cambia', height=420, legend=dict(orientation='h', y=-0.2))
fig.update_yaxes(title_text='Valor mediano (M€)', rangemode='tozero', secondary_y=False)
fig.update_yaxes(title_text='Overall promedio', range=[55, 75], secondary_y=True)
fig.update_xaxes(dtick=1)
fig.show()
print(f"Valor mediano: {fmt_eur(rollup['Valor_mediano'].iloc[0])} en {rollup.index[0]} y {fmt_eur(rollup['Valor_mediano'].iloc[-1])} en {rollup.index[-1]} "
      f"(x{rollup['Valor_mediano'].iloc[-1] / rollup['Valor_mediano'].iloc[0]:.1f}); overall promedio de {rollup['Overall_prom'].iloc[0]:.1f} a {rollup['Overall_prom'].iloc[-1]:.1f}")

# Fig. 4.7 — Slice de la temporada de mercado: valor mediano por región y grupo de edad
slice_t = (df_val[df_val['Temporada'] == TEMPORADA_MERCADO]
           .pivot_table(index='league_region', columns='GrupoEdad', values='value_eur', aggfunc='median')
           .reindex(columns=ORDEN_EDAD) / 1e6)
slice_t = slice_t.sort_values('Prime', ascending=False)
fig = mapa_calor(slice_t, '{:.2f}', 'YlGnBu', dict(x='Grupo de edad', y='', color='M€'))
fig.update_layout(title=f'Fig. 4.7 · Slice {TEMPORADA_MERCADO}: valor mediano (M€) por región y grupo de edad', height=420)
fig.show()

Valor mediano: €375K en 2015 y €1.0M en 2024 (x2.7); overall promedio de 63.8 a 67.1


**Interpretación de los resultados.**

La Fig. 4.6 es el mensaje más importante de los cubos de BI: la línea roja (valor mediano) casi se triplica entre 2015 y 2024, mientras que la línea azul (`overall` promedio) apenas se mueve. El precio subió por un cambio de escala de FIFA y no porque los jugadores sean mejores, y por eso los dashboards del Paso 4 usan una sola temporada. La Fig. 4.7 muestra dónde está hoy el valor: Europa y América tienen los valores medianos más altos, y Asia, Medio Oriente y Oceanía los más bajos.

En conclusión, la Etapa 4 se comunica con una tabla para las métricas, que se leen como cifras exactas, con gráficos que muestran cómo se comporta cada modelo, y con gráficos para las tendencias del mercado.

#### **Contratación: top 3 más eficientes por posición.**

El cuarto experimento prescriptivo no se pidió como dashboard, porque no tiene valores de entrada que el usuario cambie: la pregunta es una lista fija de jugadores. Por eso se representa con tablas, siguiendo la regla de este notebook (nombres que hay que identificar y valores exactos). Se usa la misma definición de la Etapa 4: la **eficiencia** es `overall / (value_eur / 1,000,000)`, los puntos de `overall` por cada millón de euros, sobre el mercado actual con `overall` de 75 o más. La columna *Recomendado* marca a los jugadores que están en las dos listas, que son la recomendación final de la Etapa 4.

In [32]:
# Tabla 4.2 — Contratación: top 3 más eficientes por posición (misma lógica de la Etapa 4, Paso 3.5)
OVERALL_MIN = 75
cont = snap[snap['overall'] >= OVERALL_MIN].copy()
cont['eficiencia'] = cont['overall'] / (cont['value_eur'] / 1_000_000)   # puntos de overall por millón de euros
posiciones_c = {'Defensor': 'Defensa', 'Mediocampista': 'Mediocampista', 'Delantero': 'Delantero'}

def ranking_eficiencia(datos, edad_max=None):
    filas = []
    for nombre, cat in posiciones_c.items():
        sub = datos[datos['TipoPosicion'] == cat]
        if edad_max is not None:
            sub = sub[sub['age'] <= edad_max]
        t = sub.sort_values(['eficiencia', 'overall', 'age', 'player_id'], ascending=[False, False, True, True]).head(3).copy()
        t['Posición'], t['Puesto'] = nombre, range(1, len(t) + 1)
        filas.append(t)
    return pd.concat(filas).reset_index(drop=True)

top3 = ranking_eficiencia(cont)
top3_jov = ranking_eficiencia(cont, edad_max=28)
assert (top3['Posición'].map(posiciones_c) == top3['TipoPosicion']).all()
# Recomendación de la Etapa 4: por posición, el mejor de la lista de hasta 28 años que también está en el top 3 general
recomendados = {top3_jov.loc[(top3_jov['Posición'] == p) & top3_jov['player_id'].isin(top3['player_id']), 'player_id'].iloc[0]
                for p in posiciones_c}

cols_c = ['Posición', 'Puesto', 'short_name', 'PosicionPrincipal', 'club_name', 'age', 'overall', 'value_eur', 'eficiencia', 'valor_estimado']
nombres_c = {'short_name': 'Jugador', 'PosicionPrincipal': 'Pos.', 'club_name': 'Club', 'age': 'Edad', 'overall': 'Overall',
             'value_eur': 'Valor', 'eficiencia': 'Eficiencia (overall/M€)', 'valor_estimado': 'Valor estimado'}
formatos_c = {'Valor': fmt_eur, 'Eficiencia (overall/M€)': '{:.1f}'}

for titulo, datos in [(f'Tabla 4.2 · Contratación: top 3 más eficientes por posición (overall ≥ {OVERALL_MIN}, mercado {TEMPORADA_MERCADO})', top3),
                      ('Tabla 4.3 · Lista complementaria: top 3 por posición con jugadores de hasta 28 años', top3_jov)]:
    t = datos.assign(**{'Recomendado': np.where(datos['player_id'].isin(recomendados), '✔', '')})[cols_c + ['Recomendado']]
    mostrar_tabla(t.rename(columns=nombres_c), formatos_c, titulo)
print(f"Eficiencia mediana de los {len(cont):,} jugadores con overall ≥ {OVERALL_MIN}: {cont['eficiencia'].median():.1f} puntos por millón")
print('Recomendados (uno por posición, presentes en las dos listas):', ', '.join(top3.loc[top3['player_id'].isin(recomendados), 'short_name']))


Posición,Puesto,Jugador,Pos.,Club,Edad,Overall,Valor,Eficiencia (overall/M€),Valor estimado,Recomendado
Defensor,1,Ahmed Sharahili,CB,Al Ittihad,29,75,€668K,112.4,No,
Defensor,2,Rasmus Nicolaisen,CB,Toulouse Football Club,26,76,€1.2M,65.5,No,✔
Defensor,3,Robin Pröpper,CB,FC Twente,30,75,€1.2M,64.2,No,
Mediocampista,1,Tarek Hamed,CDM,Damac FC,34,75,€1.0M,75.0,No,
Mediocampista,2,Salis Abdul Samed,CDM,Racing Club de Lens,23,78,€1.1M,73.9,No,✔
Mediocampista,3,Johan Gastien,CDM,Clermont Foot 63,35,76,€1.1M,68.6,Sí,
Delantero,1,Amahl Pellegrino,LW,FK Bodø/Glimt,33,75,€912K,82.2,Sí,
Delantero,2,Miguel Ángel Merentiel,ST,Boca Juniors,27,75,€1.1M,69.0,No,✔
Delantero,3,Thijs Dallinga,ST,Toulouse Football Club,23,75,€1.3M,57.7,No,


Posición,Puesto,Jugador,Pos.,Club,Edad,Overall,Valor,Eficiencia (overall/M€),Valor estimado,Recomendado
Defensor,1,Rasmus Nicolaisen,CB,Toulouse Football Club,26,76,€1.2M,65.5,No,✔
Defensor,2,Asger Sørensen,CB,AC Sparta Praha,27,75,€1.2M,63.4,No,
Defensor,3,Igoh Ogbu,CB,SK Slavia Praha,23,75,€1.2M,62.5,No,
Mediocampista,1,Salis Abdul Samed,CDM,Racing Club de Lens,23,78,€1.1M,73.9,No,✔
Mediocampista,2,Artem Bondarenko,CAM,Shakhtar Donetsk,23,75,€1.1M,68.2,No,
Mediocampista,3,Damian Szymański,CDM,AEK Athens,28,75,€1.3M,56.9,Sí,
Delantero,1,Miguel Ángel Merentiel,ST,Boca Juniors,27,75,€1.1M,69.0,No,✔
Delantero,2,Thijs Dallinga,ST,Toulouse Football Club,23,75,€1.3M,57.7,No,
Delantero,3,Rasmus Højlund,ST,Manchester United,20,76,€1.4M,55.8,No,


Eficiencia mediana de los 1,642 jugadores con overall ≥ 75: 9.4 puntos por millón
Recomendados (uno por posición, presentes en las dos listas): Rasmus Nicolaisen, Salis Abdul Samed, Miguel Ángel Merentiel


**Interpretación de los resultados.**

La Tabla 4.2 reproduce el top 3 de la Etapa 4 entre los 1,642 jugadores del mercado actual con `overall` de 75 o más. Ahmed Sharahili, Tarek Hamed y Amahl Pellegrino son los más eficientes de su posición (de 75 a 112 puntos de `overall` por millón, frente a una mediana de 9.4), pero tienen de 29 a 34 años. La Tabla 4.3 repite el ranking solo con jugadores de hasta 28 años.

La columna *Recomendado* resume la decisión de la Etapa 4: Rasmus Nicolaisen como defensor, Salis Abdul Samed como mediocampista y Miguel Ángel Merentiel como delantero. Los tres aparecen en las dos listas, tienen de 23 a 27 años y cuestan entre €1.1M y €1.2M. La columna *Valor estimado* advierte que el valor de Gastien, Pellegrino y Szymański se rellenó en la Etapa 2.

En conclusión, la contratación se representa con tablas porque la respuesta son nueve nombres con sus cifras exactas, y no se convirtió en dashboard porque el enunciado lo pide solo para los tres primeros experimentos.

## **Paso 4: Dashboards Interactivos del Análisis Prescriptivo**

> *En la etapa de análisis prescriptivo los 3 primeros experimentos son casos en los que el usuario podría modificar los valores de entrada, por lo que debemos: identificar cuáles son valores que el usuario podría ingresar, y crear un dashboard interactivo para responder dichos problemas según los parámetros del usuario.*

#### **Contenido de la sección:**
1. Identificación de los valores de entrada.
2. Dashboard 1: mercado de transferencias.
3. Dashboard 2: construcción de equipo.
4. Dashboard 3: vender o no vender.
5. Dashboard integrado (las tres herramientas en pestañas).

**Cómo usar los dashboards.** Hay que ejecutar todas las celdas en orden (en Colab: *Entorno de ejecución → Ejecutar todas*). Cada control recalcula el resultado al soltarlo. Los controles inician con los valores del enunciado, de modo que lo primero que se ve es la recomendación de la Etapa 4.

### **1. Identificación de los valores de entrada.**

Para cada experimento se separaron los valores que el enunciado fija y que un usuario (un director deportivo, por ejemplo) querría cambiar, de los que son parte del método y no conviene exponer. El criterio fue: **es una entrada si responde a una situación distinta del club**, como otro presupuesto, otra formación u otra oferta; **no es una entrada** si cambiarla rompe la lógica del modelo, como las columnas que se usan o el solver.

| Experimento | Valor de entrada | Valor del enunciado | Rango del control | Por qué es una entrada |
|---|---|---|---|---|
| **1. Mercado de transferencias** | Presupuesto | €100M | €10M a €300M | Cada club tiene un presupuesto distinto. |
| | Cantidad de jugadores | 5 | 1 a 11 | Depende de cuántas plazas quiere cubrir el club. |
| | Edad máxima (menor a) | 25 años | 18 a 35 | Define qué tan a largo plazo es la inversión. |
| | Potencial mínimo (mayor a) | 85 | 70 a 92 | Define qué tan exigente es el club. |
| | Media de referencia del valor por rating | Automática (Etapa 4) | Automática / todo el mercado / grupo elegible / sin filtro | El enunciado no dice qué media usar; la Etapa 4 justificó la opción automática. |
| | Qué se maximiza | Potencial | Potencial / overall / ambos | Un club que necesita rendimiento inmediato maximizaría el overall. |
| | Excluir valores estimados | No | Sí / No | Para quien solo quiere precios que vienen tal cual de FIFA. |
| **2. Construcción de equipo** | Presupuesto (costo menor a) | €150M | €20M a €400M | Cada club tiene un presupuesto distinto. |
| | Formación | 1-4-3-3 | 1-4-3-3, 1-4-4-2, 1-3-5-2, 1-5-3-2, 1-4-5-1, 1-3-4-3 | Cada entrenador juega con un sistema distinto. |
| | Método | Greedy y linprog | Ambos / Greedy / Programación lineal | Para comparar los dos enfoques que pide el enunciado. |
| | Edad máxima | Sin límite | 17 a 45 | Un club puede querer un equipo joven. |
| | Excluir valores estimados | No | Sí / No | Igual que en el experimento 1. |
| **3. Vender o no** | Edad, overall y potencial del jugador | 28, 82 y 83 | 17-40, 50-95, 50-95 | El mismo análisis sirve para cualquier jugador de la plantilla. |
| | Valor de mercado u oferta recibida | Supuesto: mediana de perfiles cercanos | €0.5M a €150M | La Etapa 4 concluyó que la decisión depende de la oferta. |
| | Salario semanal | Supuesto: percentil 85 ("salario alto") | €1K a €600K | El enunciado no da el salario exacto. |
| | Amplitud del mercado de referencia | ±2 de overall y ±2 años | ±1 a ±5 | Define con quién se compara al jugador. |
| | Criterios mínimos para vender | 2 de 3 | 1 a 3 | Representa qué tan conservador es el club. |

### **2. Dashboard 1: mercado de transferencias.**

La función `resolver_transferencias` aplica la misma lógica de la Etapa 4: arma el grupo de jóvenes elegibles, filtra los que tienen un valor por rating por debajo de la media de referencia y resuelve un problema de programación lineal entera que elige `n` jugadores maximizando el potencial total sin pasar del presupuesto. Si no hay suficientes candidatos o no existe una combinación dentro del presupuesto, el dashboard lo dice en lugar de mostrar un resultado inválido.

In [33]:
def resolver_transferencias(mercado, presupuesto=100e6, n_jugadores=5, edad_menor_a=25, potencial_mayor_a=85,
                            referencia='Automática', objetivo='Potencial', excluir_estimados=False):
    # Elige n jugadores (edad < edad_menor_a, potencial > potencial_mayor_a, valor por rating < media)
    # maximizando el objetivo sin pasar del presupuesto (programación lineal entera, igual que la Etapa 4).
    grupo = mercado[(mercado['age'] < edad_menor_a) & (mercado['potential'] > potencial_mayor_a)].copy()
    if excluir_estimados:
        grupo = grupo[grupo['valor_estimado'] == 'No']
    media_mercado = mercado['valor_por_rating'].mean()
    media_grupo = grupo['valor_por_rating'].mean() if len(grupo) else np.nan

    if referencia == 'Automática':
        usa_mercado = (grupo['valor_por_rating'] < media_mercado).sum() >= n_jugadores
        media_ref, texto_ref = (media_mercado, 'de todo el mercado') if usa_mercado else (media_grupo, 'del grupo elegible')
    elif referencia == 'Todo el mercado':
        media_ref, texto_ref = media_mercado, 'de todo el mercado'
    elif referencia == 'Grupo elegible':
        media_ref, texto_ref = media_grupo, 'del grupo elegible'
    else:
        media_ref, texto_ref = np.inf, 'sin filtro de valor por rating'

    candidatos = grupo[grupo['valor_por_rating'] < media_ref].reset_index(drop=True)
    r = dict(grupo=grupo, candidatos=candidatos, media_ref=media_ref, texto_ref=texto_ref, media_mercado=media_mercado,
             media_grupo=media_grupo, fichajes=candidatos.iloc[0:0], ok=False, mensaje='')
    if len(candidatos) < n_jugadores:
        r['mensaje'] = (f'Solo hay {len(candidatos)} candidatos que cumplen las condiciones y se pidieron {n_jugadores}. '
                        'Pruebe con una edad mayor, un potencial menor u otra media de referencia.')
        return r

    pesos = {'Potencial': (1, 0.001), 'Overall': (0.001, 1), 'Potencial + overall': (1, 1)}[objetivo]   # el peso 0.001 solo desempata
    c = -(pesos[0] * candidatos['potential'].values + pesos[1] * candidatos['overall'].values)
    sol = linprog(c, A_ub=csr_matrix(candidatos['value_eur'].values.reshape(1, -1).astype(float)), b_ub=[presupuesto],
                  A_eq=csr_matrix(np.ones((1, len(candidatos)))), b_eq=[n_jugadores],
                  bounds=(0, 1), integrality=np.ones(len(candidatos)), method='highs')
    if sol.status != 0 or sol.x is None:
        r['mensaje'] = (f'No existe una combinación de {n_jugadores} candidatos que cueste {fmt_eur(presupuesto)} o menos. '
                        f"Los {n_jugadores} más baratos cuestan {fmt_eur(candidatos['value_eur'].nsmallest(n_jugadores).sum())}.")
        return r
    fichajes = candidatos[sol.x > 0.5].sort_values(['potential', 'overall'], ascending=False)
    # Comprobación de las restricciones (igual que en la Etapa 4)
    assert len(fichajes) == n_jugadores and fichajes['player_id'].is_unique
    assert (fichajes['age'] < edad_menor_a).all() and (fichajes['potential'] > potencial_mayor_a).all()
    assert (fichajes['valor_por_rating'] < media_ref).all() and fichajes['value_eur'].sum() <= presupuesto
    r.update(fichajes=fichajes, ok=True, mensaje='Solución óptima')
    return r


# Verificación: con los valores del enunciado se reproduce el resultado de la Etapa 4
r_enunciado = resolver_transferencias(snap)
print(f"Referencia usada: media {r_enunciado['texto_ref']} ({fmt_eur(r_enunciado['media_ref'])} por punto) | candidatos: {len(r_enunciado['candidatos'])}")
print(f"Fichajes: {', '.join(r_enunciado['fichajes']['short_name'])}")
print(f"Costo: {fmt_eur(r_enunciado['fichajes']['value_eur'].sum())} | Potencial promedio: {r_enunciado['fichajes']['potential'].mean():.1f}")

Referencia usada: media del grupo elegible (€259K por punto) | candidatos: 55
Fichajes: Youssoufa Moukoko, Rayan Cherki, Fabio Miretti, Antonio Nusa, António Silva
Costo: €41.0M | Potencial promedio: 88.0


In [34]:
def figura_transferencias(r, presupuesto):
    cand, fich = r['candidatos'], r['fichajes']
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.12,
                        subplot_titles=('Candidatos elegibles: overall vs potencial', 'Costo de cada fichaje'))
    fig.add_scatter(x=cand['overall'], y=cand['potential'], mode='markers', name='Candidatos',
                    marker=dict(size=9, color=COLOR_GRIS, line=dict(width=1, color='#64748b')),
                    text=cand['short_name'], customdata=cand['value_eur'] / 1e6,
                    hovertemplate='%{text}<br>Overall %{x} · Potencial %{y}<br>€%{customdata:.1f}M<extra></extra>', row=1, col=1)
    for cat in ORDEN_POS:
        f = fich[fich['cat'] == cat]
        if len(f):
            fig.add_scatter(x=f['overall'], y=f['potential'], mode='markers', name=cat, text=f['short_name'],
                            marker=dict(size=15, color=COLOR_POS[cat], line=dict(width=1.5, color='black')),
                            customdata=f['value_eur'] / 1e6,
                            hovertemplate='<b>%{text}</b><br>Overall %{x} · Potencial %{y}<br>€%{customdata:.1f}M<extra></extra>', row=1, col=1)
    orden = fich.sort_values('value_eur')
    fig.add_bar(x=orden['value_eur'] / 1e6, y=orden['short_name'], orientation='h', showlegend=False,
                marker_color=orden['cat'].map(COLOR_POS), text=orden['value_eur'].map(fmt_eur), textposition='outside', row=1, col=2)
    fig.update_xaxes(title_text='Overall', row=1, col=1)
    fig.update_yaxes(title_text='Potencial', row=1, col=1)
    fig.update_xaxes(title_text='Valor de mercado (M€)', range=[0, max(orden['value_eur'].max() / 1e6 * 1.35, 1)], row=1, col=2)
    gasto = fich['value_eur'].sum()
    fig.update_layout(height=460, legend=dict(orientation='h', y=-0.18),
                      title=f'Fichajes recomendados: {fmt_eur(gasto)} de {fmt_eur(presupuesto)} ({gasto / presupuesto * 100:.0f}% del presupuesto)')
    return fig


def vista_transferencias(presupuesto_m, n_jugadores, edad_menor_a, potencial_mayor_a, referencia, objetivo, excluir_estimados):
    presupuesto = presupuesto_m * 1e6
    r = resolver_transferencias(snap, presupuesto, n_jugadores, edad_menor_a, potencial_mayor_a, referencia, objetivo, excluir_estimados)
    tarjetas([('Jóvenes elegibles', f"{len(r['grupo']):,}"),
              ('Candidatos (valor/rating < media)', f"{len(r['candidatos']):,}"),
              ('Media de referencia', f"{fmt_eur(r['media_ref'])}/punto" if np.isfinite(r['media_ref']) else 'Sin filtro')])
    if not r['ok']:
        aviso(r['mensaje'])
        return
    f = r['fichajes']
    gasto = f['value_eur'].sum()
    tarjetas([('Costo total', fmt_eur(gasto)), ('Presupuesto restante', fmt_eur(presupuesto - gasto)),
              ('Potencial promedio', f"{f['potential'].mean():.1f}"), ('Overall promedio', f"{f['overall'].mean():.1f}"),
              ('Edad promedio', f"{f['age'].mean():.1f}")])
    figura_transferencias(r, presupuesto).show()
    t = f[['short_name', 'club_name', 'PosicionPrincipal', 'cat', 'age', 'overall', 'potential', 'value_eur', 'valor_por_rating', 'valor_estimado']]
    t = t.rename(columns={'short_name': 'Jugador', 'club_name': 'Club', 'PosicionPrincipal': 'Posición', 'cat': 'Línea', 'age': 'Edad',
                          'overall': 'Overall', 'potential': 'Potencial', 'value_eur': 'Valor', 'valor_por_rating': 'Valor por rating',
                          'valor_estimado': 'Valor estimado'})
    mostrar_tabla(t, {'Valor': fmt_eur, 'Valor por rating': fmt_eur, 'Potencial': '{:.0f}'}, f"Fichajes recomendados (media {r['texto_ref']})")
    if (f['valor_estimado'] == 'Sí').any():
        aviso('Los jugadores con valor estimado tienen un precio rellenado en la Etapa 2: conviene confirmarlo antes de ofertar.', '#2563eb')


def crear_dashboard_transferencias():
    w = dict(
        presupuesto_m=widgets.FloatSlider(value=100, min=10, max=300, step=5, description='Presupuesto (M€)', continuous_update=False, style=ESTILO),
        n_jugadores=widgets.IntSlider(value=5, min=1, max=11, description='Jugadores a fichar', continuous_update=False, style=ESTILO),
        edad_menor_a=widgets.IntSlider(value=25, min=18, max=35, description='Edad menor a', continuous_update=False, style=ESTILO),
        potencial_mayor_a=widgets.IntSlider(value=85, min=70, max=92, description='Potencial mayor a', continuous_update=False, style=ESTILO),
        referencia=widgets.Dropdown(options=['Automática', 'Todo el mercado', 'Grupo elegible', 'Sin filtro'], value='Automática',
                                    description='Media de referencia', style=ESTILO),
        objetivo=widgets.Dropdown(options=['Potencial', 'Overall', 'Potencial + overall'], value='Potencial', description='Maximizar', style=ESTILO),
        excluir_estimados=widgets.Checkbox(value=False, description='Excluir valores estimados', style=ESTILO),
    )
    salida = widgets.interactive_output(vista_transferencias, w)
    controles = widgets.HBox([widgets.VBox([w['presupuesto_m'], w['n_jugadores'], w['edad_menor_a'], w['potencial_mayor_a']]),
                              widgets.VBox([w['referencia'], w['objetivo'], w['excluir_estimados']])])
    titulo = widgets.HTML(f'<h3 style="margin:4px 0">Mercado de transferencias · temporada {TEMPORADA_MERCADO}</h3>'
                          '<div>Elige los jugadores que maximizan el potencial total sin pasar del presupuesto.</div>')
    return widgets.VBox([titulo, controles, salida])


display(crear_dashboard_transferencias())

**Interpretación de los resultados.**

Con los valores del enunciado, el dashboard muestra la misma recomendación de la Etapa 4: cinco jóvenes de potencial 87 a 89 por unos €41 millones, con más de la mitad del presupuesto sin usar. Las tarjetas superiores explican de dónde sale la lista (cuántos jóvenes cumplen la edad y el potencial, y cuántos pasan el filtro de valor por rating) y las inferiores resumen la compra.

El dashboard permite responder preguntas que la Etapa 4 no podía responder con un único resultado. Por ejemplo, al bajar el presupuesto se ve a partir de qué monto cambia la selección (el presupuesto deja de ser holgado), al cambiar *Maximizar* a *Overall* la lista pasa a jugadores más hechos, y al elegir *Todo el mercado* como referencia aparece el aviso de que no hay cinco candidatos, que es la razón por la que la Etapa 4 eligió la media del grupo.

En conclusión, la recomendación de la Etapa 4 es un caso particular del dashboard, y el usuario puede adaptarla a su propio presupuesto y criterio.

### **3. Dashboard 2: construcción de equipo.**

Se reutilizan los dos métodos de la Etapa 4. El **greedy** recorre los jugadores de mayor a menor `overall` y ficha a cada uno si queda cupo en su posición y si el dinero restante alcanza para completar las posiciones pendientes con los jugadores más baratos. La **programación lineal entera** (`linprog` con HiGHS) elige la combinación exacta que maximiza el `overall` total. El costo debe ser **menor** al presupuesto, por lo que el límite es el presupuesto menos €1, como en la Etapa 4. El equipo se dibuja sobre una cancha para que la formación se vea de inmediato.

In [35]:
FORMACIONES = {
    '1-4-3-3 (enunciado)': {'Portero': 1, 'Defensa': 4, 'Mediocampista': 3, 'Delantero': 3},
    '1-4-4-2': {'Portero': 1, 'Defensa': 4, 'Mediocampista': 4, 'Delantero': 2},
    '1-3-5-2': {'Portero': 1, 'Defensa': 3, 'Mediocampista': 5, 'Delantero': 2},
    '1-5-3-2': {'Portero': 1, 'Defensa': 5, 'Mediocampista': 3, 'Delantero': 2},
    '1-4-5-1': {'Portero': 1, 'Defensa': 4, 'Mediocampista': 5, 'Delantero': 1},
    '1-3-4-3': {'Portero': 1, 'Defensa': 3, 'Mediocampista': 4, 'Delantero': 3},
}
ORDEN_LINEA = {c: i for i, c in enumerate(ORDEN_POS)}


def greedy_equipo(pool, req, limite):
    # Misma lógica que la Etapa 4 (se recorre con itertuples para que el dashboard responda más rápido).
    restantes, gastado, elegidos, ids_elegidos = dict(req), 0.0, [], set()
    baratos = {c: pool[pool['cat'] == c].nsmallest(max(req.values()) + 5, 'value_eur')[['player_id', 'value_eur']].values.tolist() for c in req}

    def minimo_pendiente(rest):
        total = 0
        for c, n in rest.items():
            usados = 0
            for pid, costo in baratos[c]:
                if usados == n:
                    break
                if pid not in ids_elegidos:
                    total += costo
                    usados += 1
        return total

    orden = pool.sort_values(['overall', 'value_eur'], ascending=[False, True])
    for j in orden.itertuples():
        c = j.cat
        if c not in restantes or restantes[c] == 0:
            continue
        nuevo = dict(restantes)
        nuevo[c] -= 1
        ids_elegidos.add(j.player_id)
        if gastado + j.value_eur + minimo_pendiente(nuevo) <= limite:
            restantes, gastado = nuevo, gastado + j.value_eur
            elegidos.append(j.Index)
        else:
            ids_elegidos.discard(j.player_id)
        if sum(restantes.values()) == 0:
            break
    return pool.loc[elegidos].copy() if sum(restantes.values()) == 0 else None


def lp_equipo(pool, req, limite):
    cats = list(req)
    sol = linprog(-pool['overall'].values.astype(float),
                  A_ub=csr_matrix(pool['value_eur'].values.reshape(1, -1).astype(float)), b_ub=[limite],
                  A_eq=csr_matrix(np.array([(pool['cat'] == c).values.astype(float) for c in cats])), b_eq=[req[c] for c in cats],
                  bounds=(0, 1), integrality=np.ones(len(pool)), method='highs')
    if sol.status != 0 or sol.x is None:
        return None
    return pool[sol.x > 0.5].copy()


def construir_equipo(mercado, presupuesto=150e6, formacion='1-4-3-3 (enunciado)', metodo='Ambos', edad_max=45, excluir_estimados=False):
    req = FORMACIONES[formacion]
    pool = mercado[mercado['age'] <= edad_max]
    if excluir_estimados:
        pool = pool[pool['valor_estimado'] == 'No']
    pool = pool.reset_index(drop=True)
    limite = presupuesto - 1   # costo MENOR al presupuesto
    minimo = sum(pool[pool['cat'] == c]['value_eur'].nsmallest(n).sum() for c, n in req.items())
    faltan = {c: n for c, n in req.items() if (pool['cat'] == c).sum() < n}
    r = dict(req=req, pool=pool, limite=limite, minimo=minimo, equipos={}, mensaje='')
    if faltan:
        r['mensaje'] = f'No hay suficientes jugadores con esos filtros en: {", ".join(faltan)}.'
        return r
    if minimo > limite:
        r['mensaje'] = f'El equipo más barato posible cuesta {fmt_eur(minimo)}: el presupuesto no alcanza.'
        return r
    if metodo in ('Ambos', 'Greedy'):
        r['equipos']['Greedy'] = greedy_equipo(pool, req, limite)
    if metodo in ('Ambos', 'Programación lineal'):
        r['equipos']['Programación lineal'] = lp_equipo(pool, req, limite)
    for nombre, eq in r['equipos'].items():
        if eq is None:
            continue
        eq.sort_values(['cat', 'overall'], key=lambda s: s.map(ORDEN_LINEA) if s.name == 'cat' else -s, inplace=True)
        # Comprobación de las restricciones (igual que en la Etapa 4)
        assert len(eq) == sum(req.values()) and eq['player_id'].is_unique
        assert eq['cat'].value_counts().to_dict() == {c: n for c, n in req.items() if n > 0}
        assert eq['value_eur'].sum() < presupuesto
    return r


# Verificación: con los valores del enunciado se reproduce el resultado de la Etapa 4
r_eq = construir_equipo(snap)
for nombre, eq in r_eq['equipos'].items():
    print(f"{nombre}: rating total {int(eq['overall'].sum())} | costo {fmt_eur(eq['value_eur'].sum())} | "
          f"jugadores: {', '.join(eq['short_name'])}")

Greedy: rating total 832 | costo €150.0M | jugadores: Ivan Provedel, Javi Galán, Dennis Hadžikadunić, Hasan Kuruçay, Liu Tao, Jaminton Campaz, Dominik Schwizer, Sabit Abdusalam, Erling Haaland, Karim Benzema, Victor Osimhen
Programación lineal: rating total 932 | costo €145.6M | jugadores: Gregor Kobel, Giovanni Di Lorenzo, Kim Min Jae, Federico Dimarco, Javi Galán, Martin Ødegaard, Stanislav Lobotka, André-Franck Zambo Anguissa, Victor Osimhen, Rafael Leão, Randal Kolo Muani


In [36]:
def figura_cancha(equipos):
    # Dibuja cada equipo sobre una cancha (portero abajo, delanteros arriba)
    nombres = list(equipos)
    fig = make_subplots(rows=1, cols=len(nombres), subplot_titles=[
        f"{n}: rating {int(e['overall'].sum())} · {fmt_eur(e['value_eur'].sum())}" for n, e in equipos.items()], horizontal_spacing=0.04)
    altura = {'Portero': 0.10, 'Defensa': 0.32, 'Mediocampista': 0.65, 'Delantero': 0.83}
    for k, (nombre, eq) in enumerate(equipos.items(), start=1):
        lineas = [('rect', 0, 0, 1, 1), ('line', 0, 0.5, 1, 0.5), ('rect', 0.3, 0, 0.7, 0.04), ('rect', 0.3, 0.96, 0.7, 1)]
        for tipo, x0, y0, x1, y1 in lineas:
            fig.add_shape(type=tipo, x0=x0, y0=y0, x1=x1, y1=y1, line=dict(color='white', width=2),
                          fillcolor='#15803d' if (tipo, x0) == ('rect', 0) else None, layer='below', row=1, col=k)
        fig.add_shape(type='circle', x0=0.42, y0=0.45, x1=0.58, y1=0.55, line=dict(color='white', width=2), layer='below', row=1, col=k)
        for cat in ORDEN_POS:
            linea = eq[eq['cat'] == cat]
            xs = [(i + 1) / (len(linea) + 1) for i in range(len(linea))]
            fig.add_scatter(x=xs, y=[altura[cat]] * len(linea), mode='markers+text', showlegend=(k == 1), name=cat, legendgroup=cat,
                            marker=dict(size=34, color=COLOR_POS[cat], line=dict(color='white', width=2)),
                            text=linea['overall'].astype(int).astype(str), textfont=dict(color='white', size=13),
                            customdata=np.stack([linea['short_name'], linea['club_name'], linea['value_eur'].map(fmt_eur), linea['age']], axis=-1),
                            hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<br>Overall %{text} · %{customdata[3]} años<br>%{customdata[2]}<extra></extra>',
                            row=1, col=k)
            for x, nombre_j in zip(xs, linea['short_name']):
                fig.add_annotation(x=x, y=altura[cat] - 0.075, text=nombre_j, showarrow=False, font=dict(color='white', size=10), row=1, col=k)
        fig.update_xaxes(range=[-0.02, 1.02], visible=False, row=1, col=k)
        fig.update_yaxes(range=[-0.02, 1.02], visible=False, row=1, col=k)
    fig.update_layout(height=560, plot_bgcolor='white', legend=dict(orientation='h', y=-0.05), margin=dict(t=60, l=10, r=10, b=30),
                      title='Equipo en la cancha (el número es el overall; pase el cursor para ver club y valor)')
    return fig


def vista_equipo(presupuesto_m, formacion, metodo, edad_max, excluir_estimados):
    presupuesto = presupuesto_m * 1e6
    r = construir_equipo(snap, presupuesto, formacion, metodo, edad_max, excluir_estimados)
    if r['mensaje']:
        aviso(r['mensaje'])
        return
    equipos = {n: e for n, e in r['equipos'].items() if e is not None}
    if not equipos:
        aviso('Ningún método encontró un equipo válido con estos parámetros.')
        return
    for nombre, eq in equipos.items():
        gasto = eq['value_eur'].sum()
        display(HTML(f'<b>{nombre}</b>'))
        tarjetas([('Rating total', f"{int(eq['overall'].sum())}"), ('Rating promedio', f"{eq['overall'].mean():.2f}"),
                  ('Costo', fmt_eur(gasto)), ('Presupuesto sin usar', fmt_eur(presupuesto - gasto)), ('Edad promedio', f"{eq['age'].mean():.1f}")])
    figura_cancha(equipos).show()

    if len(equipos) == 2:
        g, l = equipos['Greedy'], equipos['Programación lineal']
        comp = pd.DataFrame({'Greedy': g.groupby('cat')['overall'].mean(), 'Programación lineal': l.groupby('cat')['overall'].mean()}).reindex(
            [c for c in ORDEN_POS if r['req'].get(c)])
        fig = go.Figure([go.Bar(name=m, x=comp.index, y=comp[m], text=comp[m].map(lambda v: f'{v:.1f}'), textposition='outside', marker_color=col)
                         for m, col in [('Greedy', '#64748b'), ('Programación lineal', '#16a34a')]])
        fig.update_layout(barmode='group', height=360, title='Overall promedio por línea: greedy vs programación lineal',
                          yaxis=dict(range=[max(0, comp.min().min() - 10), comp.max().max() + 4], title='Overall promedio'))
        fig.show()
        dif = int(l['overall'].sum() - g['overall'].sum())
        comunes = len(set(g['player_id']) & set(l['player_id']))
        aviso(f'La programación lineal logra {dif:+d} puntos de rating frente al greedy y gasta {fmt_eur(l.value_eur.sum() - g.value_eur.sum())} '
              f'{"más" if l.value_eur.sum() > g.value_eur.sum() else "menos"}. Jugadores en común: {comunes}.', '#16a34a')

    for nombre, eq in equipos.items():
        t = eq[['short_name', 'cat', 'PosicionPrincipal', 'club_name', 'age', 'overall', 'value_eur', 'valor_estimado']].rename(columns={
            'short_name': 'Jugador', 'cat': 'Línea', 'PosicionPrincipal': 'Posición', 'club_name': 'Club', 'age': 'Edad', 'overall': 'Overall',
            'value_eur': 'Valor', 'valor_estimado': 'Valor estimado'})
        mostrar_tabla(t, {'Valor': fmt_eur}, f'Plantilla · {nombre}')


def crear_dashboard_equipo():
    w = dict(
        presupuesto_m=widgets.FloatSlider(value=150, min=20, max=400, step=5, description='Presupuesto (M€)', continuous_update=False, style=ESTILO),
        formacion=widgets.Dropdown(options=list(FORMACIONES), value='1-4-3-3 (enunciado)', description='Formación', style=ESTILO),
        metodo=widgets.ToggleButtons(options=['Ambos', 'Greedy', 'Programación lineal'], value='Ambos', description='Método', style=ESTILO),
        edad_max=widgets.IntSlider(value=45, min=17, max=45, description='Edad máxima', continuous_update=False, style=ESTILO),
        excluir_estimados=widgets.Checkbox(value=False, description='Excluir valores estimados', style=ESTILO),
    )
    salida = widgets.interactive_output(vista_equipo, w)
    controles = widgets.VBox([widgets.HBox([w['presupuesto_m'], w['formacion']]), widgets.HBox([w['edad_max'], w['excluir_estimados']]), w['metodo']])
    titulo = widgets.HTML(f'<h3 style="margin:4px 0">Construcción de equipo · temporada {TEMPORADA_MERCADO}</h3>'
                          '<div>Arma el equipo de mayor rating total con un costo menor al presupuesto.</div>')
    return widgets.VBox([titulo, controles, salida])


display(crear_dashboard_equipo())

**Interpretación de los resultados.**

Con los valores del enunciado, el dashboard reproduce los dos equipos de la Etapa 4: la programación lineal llega a un rating total de 932 por unos €145.6 millones, y el greedy a 832 por casi €150 millones. La cancha hace visible el problema del greedy sin necesidad de leer la tabla: sus delanteros tienen 88 a 91 de `overall`, pero la defensa y el mediocampo tienen círculos de 60 a 70, mientras que el equipo de `linprog` tiene todos los círculos en 82 o más.

El dashboard agrega preguntas nuevas. Al subir el presupuesto, la diferencia entre los dos métodos se reduce, porque el greedy deja de quedarse sin dinero; al bajarlo, el greedy empeora más rápido. Al cambiar la formación se ve qué línea es más cara de reforzar, y con la *Edad máxima* se puede armar, por ejemplo, el mejor equipo sub-23 posible con el mismo presupuesto.

En conclusión, el dashboard confirma con cualquier combinación de parámetros que la programación lineal es el método recomendado, y permite al usuario armar el equipo para su propio presupuesto y sistema de juego.

### **4. Dashboard 3: vender o no vender.**

La función `evaluar_venta` aplica los tres criterios de la Etapa 4. Se compara al jugador con un mercado de referencia (jugadores de la temporada actual con un `overall` y una edad parecidos) y cada criterio vota a favor de vender o de retener:

1. **Valor por rating** mayor o igual a la mediana del mercado → el mercado lo paga bien, buen momento para vender.
2. **Brecha de potencial** menor o igual a la mediana → no tiene más margen de crecimiento que sus pares.
3. **Salario** mayor que la mediana → es un costo alto para el club.

Si el usuario no ingresa una oferta ni un salario, se usan los mismos supuestos de la Etapa 4 (mediana de los perfiles más cercanos y percentil 85 del salario). El gráfico de la derecha muestra cuántos criterios a favor de vender hay para cada oferta posible, y marca el **precio de equilibrio**: la oferta a partir de la cual el criterio de valor por rating cambia a vender.

In [37]:
def evaluar_venta(mercado, edad=28, overall=82, potencial=83, oferta=None, salario=None,
                  rango_overall=2, rango_edad=2, votos_minimos=2):
    similares = mercado[mercado['overall'].between(overall - rango_overall, overall + rango_overall)
                        & mercado['age'].between(edad - rango_edad, edad + rango_edad)].copy()
    r = dict(similares=similares, ok=False, mensaje='')
    if len(similares) < 5:
        r['mensaje'] = (f'Solo hay {len(similares)} jugadores parecidos en el mercado: amplíe el rango de overall o de edad '
                        'para tener una comparación confiable.')
        return r
    similares['brecha'] = similares['potential'] - similares['overall']
    cercanos = similares[(similares['overall'] == overall) & similares['age'].between(edad - 1, edad + 1)]
    if len(cercanos) == 0:
        cercanos = similares
    valor = oferta if oferta is not None else cercanos['value_eur'].median()
    sal = salario if salario is not None else similares['wage_eur'].quantile(0.85)
    brecha = potencial - overall
    med_vpr, med_sal, med_brecha = similares['valor_por_rating'].median(), similares['wage_eur'].median(), similares['brecha'].median()
    c1, c2, c3 = valor / overall >= med_vpr, brecha <= med_brecha, sal > med_sal
    votos = int(c1) + int(c2) + int(c3)
    precio_equilibrio = med_vpr * overall
    ofertas = np.linspace(0, max(precio_equilibrio, valor) * 2, 200)
    votos_curva = (ofertas / overall >= med_vpr).astype(int) + int(c2) + int(c3)
    r.update(ok=True, cercanos=cercanos, valor=valor, salario=sal, brecha=brecha, med_vpr=med_vpr, med_sal=med_sal, med_brecha=med_brecha,
             criterios=[c1, c2, c3], votos=votos, decision='VENDER' if votos >= votos_minimos else 'RETENER',
             precio_equilibrio=precio_equilibrio, ofertas=ofertas, votos_curva=votos_curva, votos_minimos=votos_minimos,
             origen_valor='Oferta ingresada' if oferta is not None else f'Supuesto: mediana de {len(cercanos)} perfiles cercanos',
             origen_salario='Salario ingresado' if salario is not None else 'Supuesto: percentil 85 del mercado de referencia')
    return r


# Verificación: con los valores del enunciado se reproduce el resultado de la Etapa 4
r_v = evaluar_venta(snap)
print(f"Mercado de referencia: {len(r_v['similares'])} jugadores | valor supuesto {fmt_eur(r_v['valor'])} | salario supuesto {fmt_eur(r_v['salario'])}")
print(f"Criterios a favor de vender: {r_v['votos']} de 3 -> {r_v['decision']} | precio de equilibrio: {fmt_eur(r_v['precio_equilibrio'])}")

Mercado de referencia: 176 jugadores | valor supuesto €16.2M | salario supuesto €67K
Criterios a favor de vender: 2 de 3 -> VENDER | precio de equilibrio: €16.1M


In [38]:
def figura_venta(r, overall):
    c1, c2, c3 = r['criterios']
    fig = make_subplots(rows=1, cols=4, column_widths=[0.18, 0.18, 0.18, 0.46], horizontal_spacing=0.07,
                        subplot_titles=('Valor por rating (K€/punto)', 'Brecha de potencial', 'Salario semanal (K€)', 'Criterios a favor de vender según la oferta'))
    barras = [(1, r['valor'] / overall / 1e3, r['med_vpr'] / 1e3, c1, '{:,.0f}'),
              (2, r['brecha'], r['med_brecha'], c2, '{:.1f}'),
              (3, r['salario'] / 1e3, r['med_sal'] / 1e3, c3, '{:,.0f}')]
    for col, jugador, mediana, criterio, fmt in barras:
        fig.add_bar(x=['Jugador', 'Mediana'], y=[jugador, mediana], showlegend=False,
                    marker_color=[COLOR_VENDER if criterio else COLOR_RETENER, COLOR_GRIS],
                    text=[fmt.format(jugador), fmt.format(mediana)], textposition='outside', row=1, col=col)
        fig.update_yaxes(range=[0, max(jugador, mediana, 0.5) * 1.35], row=1, col=col)
    fig.add_scatter(x=r['ofertas'] / 1e6, y=r['votos_curva'], mode='lines', line=dict(shape='hv', color='#0f172a', width=3),
                    name='Criterios a favor', showlegend=False, hovertemplate='Oferta €%{x:.1f}M → %{y} de 3<extra></extra>', row=1, col=4)
    fig.add_hrect(y0=r['votos_minimos'] - 0.5, y1=3.4, fillcolor=COLOR_VENDER, opacity=0.08, line_width=0, row=1, col=4)
    fig.add_vline(x=r['precio_equilibrio'] / 1e6, line_dash='dash', line_color=COLOR_VENDER, row=1, col=4,
                  annotation_text=f"Equilibrio {fmt_eur(r['precio_equilibrio'])}", annotation_position='bottom right')
    fig.add_scatter(x=[r['valor'] / 1e6], y=[r['votos']], mode='markers', marker=dict(size=16, color=COLOR_VENDER if r['decision'] == 'VENDER' else COLOR_RETENER,
                    line=dict(color='black', width=1.5)), showlegend=False, hovertemplate='Valor/oferta actual: €%{x:.1f}M<extra></extra>', row=1, col=4)
    fig.update_xaxes(title_text='Oferta (M€)', row=1, col=4)
    fig.update_yaxes(range=[-0.2, 3.4], dtick=1, row=1, col=4)
    fig.update_layout(height=420, title='Rojo = el criterio apunta a vender · azul = apunta a retener · zona sombreada = se recomienda vender',
                      margin=dict(t=80, l=10, r=10, b=40))
    return fig


def vista_venta(edad, overall, potencial, usar_oferta, oferta_m, usar_salario, salario_k, rango_overall, rango_edad, votos_minimos):
    if potencial < overall:
        aviso('El potencial no puede ser menor que el overall (en FIFA el potencial es el techo del jugador).')
        return
    r = evaluar_venta(snap, edad, overall, potencial, oferta_m * 1e6 if usar_oferta else None, salario_k * 1e3 if usar_salario else None,
                      rango_overall, rango_edad, votos_minimos)
    if not r['ok']:
        aviso(r['mensaje'])
        return
    color = COLOR_VENDER if r['decision'] == 'VENDER' else COLOR_RETENER
    display(HTML(f"<div style='background:{color};color:white;border-radius:8px;padding:12px 16px;margin:6px 0;font-size:18px'>"
                 f"<b>{r['decision']}</b> · {r['votos']} de 3 criterios a favor de vender (se exigen {votos_minimos}) · "
                 f"vender conviene con ofertas desde <b>{fmt_eur(r['precio_equilibrio'])}</b></div>"))
    tarjetas([('Mercado de referencia', f"{len(r['similares'])} jugadores"), ('Valor u oferta', fmt_eur(r['valor'])),
              ('Salario semanal', fmt_eur(r['salario'])), ('Brecha de potencial', f"{r['brecha']}")])
    figura_venta(r, overall).show()
    nombres = ['Valor por rating ≥ mediana', 'Brecha ≤ mediana', 'Salario > mediana']
    origen = [r['origen_valor'], 'Dato ingresado (potencial − overall)', r['origen_salario']]
    t = pd.DataFrame({'Criterio': nombres,
                      'Jugador': [fmt_eur(r['valor'] / overall), f"{r['brecha']:.0f}", fmt_eur(r['salario'])],
                      'Mediana del mercado': [fmt_eur(r['med_vpr']), f"{r['med_brecha']:.1f}", fmt_eur(r['med_sal'])],
                      'Apunta a': ['Vender' if c else 'Retener' for c in r['criterios']], 'Origen del dato': origen})
    mostrar_tabla(t, titulo='Detalle de los criterios')


def crear_dashboard_venta():
    w = dict(
        edad=widgets.IntSlider(value=28, min=17, max=40, description='Edad', continuous_update=False, style=ESTILO),
        overall=widgets.IntSlider(value=82, min=50, max=95, description='Overall', continuous_update=False, style=ESTILO),
        potencial=widgets.IntSlider(value=83, min=50, max=95, description='Potencial', continuous_update=False, style=ESTILO),
        usar_oferta=widgets.Checkbox(value=False, description='Ingresar oferta (si no, se supone)', style=ESTILO),
        oferta_m=widgets.FloatSlider(value=16, min=0.5, max=150, step=0.5, description='Oferta (M€)', continuous_update=False, style=ESTILO),
        usar_salario=widgets.Checkbox(value=False, description='Ingresar salario (si no, se supone)', style=ESTILO),
        salario_k=widgets.FloatSlider(value=67, min=1, max=600, step=1, description='Salario semanal (K€)', continuous_update=False, style=ESTILO),
        rango_overall=widgets.IntSlider(value=2, min=1, max=5, description='Mercado: ± overall', continuous_update=False, style=ESTILO),
        rango_edad=widgets.IntSlider(value=2, min=1, max=5, description='Mercado: ± años', continuous_update=False, style=ESTILO),
        votos_minimos=widgets.IntSlider(value=2, min=1, max=3, description='Criterios para vender', continuous_update=False, style=ESTILO),
    )
    salida = widgets.interactive_output(vista_venta, w)
    controles = widgets.HBox([
        widgets.VBox([widgets.HTML('<b>Jugador</b>'), w['edad'], w['overall'], w['potencial']]),
        widgets.VBox([widgets.HTML('<b>Dinero</b>'), w['usar_oferta'], w['oferta_m'], w['usar_salario'], w['salario_k']]),
        widgets.VBox([widgets.HTML('<b>Método</b>'), w['rango_overall'], w['rango_edad'], w['votos_minimos']]),
    ])
    titulo = widgets.HTML(f'<h3 style="margin:4px 0">¿Vender o retener? · mercado {TEMPORADA_MERCADO}</h3>'
                          '<div>Compara al jugador con sus pares en valor por rating, brecha de potencial y salario.</div>')
    return widgets.VBox([titulo, controles, salida])


display(crear_dashboard_venta())

**Interpretación de los resultados.**

Con los valores del enunciado (28 años, `overall` 82, potencial 83 y los supuestos de valor y salario de la Etapa 4), el dashboard llega a la misma conclusión: dos de tres criterios apuntan a vender, y el precio de equilibrio es de unos €16 millones. El gráfico de la derecha es el que mejor resume la decisión: por debajo del precio de equilibrio solo el salario apunta a vender y la recomendación es retener; por encima, el valor por rating se suma y la recomendación cambia a vender.

Este es el aporte principal del dashboard. La Etapa 4 tuvo que dar una recomendación condicional porque el enunciado no da el valor ni el salario del jugador; aquí, el usuario marca *Ingresar oferta* y escribe la oferta real que recibió, y el dashboard le dice si vender. El mismo análisis sirve para cualquier otro jugador de la plantilla con solo cambiar la edad, el `overall` y el potencial; por ejemplo, un jugador de 21 años con potencial alto tiene una brecha mayor que sus pares y el segundo criterio pasa a retener.

En conclusión, el dashboard convierte la recomendación condicional de la Etapa 4 en una herramienta de decisión: dada una oferta concreta, responde vender o retener y muestra por qué.

### **5. Dashboard integrado (las tres herramientas en pestañas).**

Para la presentación se reúnen los tres dashboards en un solo panel con pestañas, así se puede pasar de un experimento a otro sin desplazarse por el notebook. Cada pestaña es una copia independiente de los dashboards anteriores.

In [39]:
pestanas = widgets.Tab(children=[crear_dashboard_transferencias(), crear_dashboard_equipo(), crear_dashboard_venta()])
for i, titulo in enumerate(['1. Transferencias', '2. Construcción de equipo', '3. Vender o no']):
    pestanas.set_title(i, titulo)
display(widgets.HTML(f'<h2 style="margin:4px 0">Dashboard prescriptivo FIFA · mercado {TEMPORADA_MERCADO}</h2>'), pestanas)

HTML(value='<h2 style="margin:4px 0">Dashboard prescriptivo FIFA · mercado 2024</h2>')

## **Paso 5: Presentación y Conclusiones**

### **1. Guion de la presentación.**

La exposición se hace con la **presentación interactiva** (https://josee-arrietaa-31.github.io/etapa5-fifa/), que sigue el recorrido "de datos a información" del proyecto: cada sección tiene un solo mensaje, y sus gráficos y dashboards usan los mismos datos y la misma lógica de este notebook. Se avanza con la flecha hacia abajo o la barra espaciadora, y los gráficos 3D se giran arrastrándolos.

| # | Sección de la presentación | Mensaje | Respaldo en este notebook |
|---|---|---|---|
| 00 | Portada | Etapa 5: de datos a información. | Introducción |
| 01 | El problema | ¿Cómo ayudar a un club a decidir fichajes, ventas y alineaciones con diez temporadas de datos de FIFA? | Introducción |
| 02 | Calidad de los datos | 10 temporadas, sin duplicados, con los registros dudosos identificados. | Tabla 1.1 · Fig. 1.1 |
| 03 | Modelo dimensional (torres 3D) | Europa concentra el 84.7 % del valor; la Premier League es la liga de mayor valor. | Figs. 2.1, 2.2 y 2.4 |
| 04 | Descriptivo: la edad | Los jóvenes tienen 13 puntos de margen de crecimiento; los veteranos ya llegaron a su techo. | Explorador (Fig. 3.1) · Fig. 3.4 |
| 05 | Roll-up por temporada | El valor mediano casi se triplicó entre 2015 y 2024 sin que subiera el nivel; por eso las decisiones usan solo el mercado de 2024. | Fig. 4.6 |
| 06 | Modelos predictivos | El valor se predice bien (R² 0.904) y el Random Forest detecta el alto potencial (F1 0.74); el salario depende de datos que no tenemos. | Tabla 4.1 · Figs. 4.2 a 4.4 |
| 07 | Tipos de jugador (3D) | K-Means encuentra cuatro perfiles, que se ven separados en tres componentes principales. | Figs. 4.1 y 4.5 |
| 08 | Valores de entrada | Qué puede cambiar el usuario en cada experimento y por qué. | Paso 4.1 |
| 09 · 10 · 11 | Dashboards en vivo | Transferencias, construcción de equipo y vender o no: se cambian el presupuesto, la formación y la oferta, y se muestra cómo cambia la recomendación. | Pasos 4.2 a 4.5 |
| 12 | Conclusiones | Lo que deja el proyecto y sus limitaciones. | Paso 5.2 |
| 13 | Anexo: explorar los datos | Distribuciones, rankings y anomalías para las preguntas del jurado. | Paso 3.3 |

Si la presentación no estuviera disponible durante la exposición, los mismos dashboards se pueden mostrar desde este notebook (panel con pestañas del Paso 4.5).

### **2. Conclusiones generales.**

1. **Inventario.** Se identificaron las preguntas de negocio respondidas en las cuatro etapas anteriores y se eligió una representación para cada una con una regla explícita: tabla cuando importan los nombres o los valores exactos, gráfico cuando importa la comparación, la forma o la tendencia.
2. **Representación.** Los gráficos interactivos permiten leer el valor exacto al pasar el cursor, hacer zoom y ocultar series, y dos herramientas (el explorador de distribuciones y el ranking interactivo) reemplazan varias decenas de gráficos estáticos de la Etapa 3.
3. **Dashboards.** Los tres primeros experimentos prescriptivos se convirtieron en dashboards con los valores de entrada que un club querría cambiar. Con los valores del enunciado, cada dashboard reproduce el resultado de la Etapa 4, y el código verifica las restricciones antes de mostrar una recomendación.
4. **Valor agregado de la interactividad.** El caso de vender o no es el que más gana: la Etapa 4 solo podía dar una recomendación condicional porque el enunciado no da el valor del jugador, y el dashboard la convierte en una respuesta concreta para la oferta que el usuario ingrese.

**Limitaciones.** Los dashboards heredan las limitaciones de las etapas anteriores: el valor de mercado es una estimación de FIFA y no un precio real de transferencia, cerca de un tercio de los valores de 2024 fueron estimados (los dashboards lo indican en cada jugador) y el dataset no incluye lesiones, contratos ni cláusulas. Además, los widgets necesitan un kernel activo: al abrir el notebook sin ejecutarlo, los dashboards no se muestran y hay que ejecutar todas las celdas.